# V_FINAL: Last Submission — Predict ALL 120 Perturbations

**Date:** March 2026  
**Base:** V20 RUN=3 (Blind MLP + cosine_light + 3-tier PB + 3x LOO, **LB=4.08996** — ALL-TIME BEST)  
**Target:** Maximize final ranking score on pert_61–120

---

### CRITICAL CONTEXT

The organizers released `pert_ids_all.csv` — the identities of ALL 120 perturbations are now known:
- **pert_1–60 (public LB):** Scored during competition. Our best = V20 RUN=3 (LB=4.090)
- **pert_61–120 (FINAL ranking):** Only these matter for the final leaderboard

**ALL existing submissions have ZEROS for rows 61–120.** We must generate real predictions.

### Strategy — Two Runs + Ensemble

| Run | Strategy | Rationale |
|-----|----------|-----------|
| **A** | Pure V20 RUN=3 for all 120 perts | Lowest risk, proven best on public LB |
| **B** | V20 RUN=3 + Replogle K562 augmentation | Regularization upside (V23 R5: LB=4.064) |
| **C** | 50/50 ensemble of A + B | Hedged bet |

### What is FROZEN (same as V20 RUN=3)
- **Architecture:** V17 LightMLP (Embedding → 2-layer MLP → 5127, H=384)
- **Loss:** cosine_light (λ=0.08, cos_right=0.0405)
- **3-Tier PseudoBulk:** T1=10 full, T2=per-channel, T3=5 half-cell
- **LOO:** Baseline 5th percentile, 3x resamples
- **KO Correction:** knockout_multiplier=1.0
- **No tricks:** No NN-matching, no weighting, no thresholding

### Checkpoint/Resume
All stages checkpoint to `/content/drive/MyDrive/myllia_vfinal/`. Colab crashes resume from last completed stage.

In [ ]:
# ============================================================
# CELL 1: Setup & Drive Mount + V_FINAL Checkpoint Directory
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/Myllia_Challenge'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
VFINAL_DIR = '/content/drive/MyDrive/myllia_vfinal'

for d in [BASE_DIR, DATA_DIR, OUTPUT_DIR, VFINAL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Base: {BASE_DIR}')
print(f'V_FINAL checkpoints: {VFINAL_DIR}')
existing = [f for f in os.listdir(VFINAL_DIR) if not f.startswith('.')]
if existing:
    print(f'  Found {len(existing)} existing checkpoint files:')
    for f in sorted(existing)[:30]:
        size = os.path.getsize(os.path.join(VFINAL_DIR, f))
        print(f'    {f} ({size/1024:.0f} KB)')
else:
    print('  No existing checkpoints (fresh start)')

# Check for V23 checkpoints (we may reuse LOO cache)
V23_DIR = '/content/drive/MyDrive/myllia_v23'
if os.path.exists(V23_DIR):
    v23_files = [f for f in os.listdir(V23_DIR) if not f.startswith('.')]
    print(f'\nV23 checkpoint dir found: {len(v23_files)} files')
    if os.path.exists(os.path.join(V23_DIR, 'loo_baseline.npz')):
        print('  ✓ LOO baseline cache available (will reuse)')
    if os.path.exists(os.path.join(V23_DIR, 'replogle_augmentation.npz')):
        print('  ✓ Replogle augmentation cache available (will reuse)')
else:
    print('\nNo V23 checkpoints found (will generate from scratch)')

print('\nDirectories ready ✓')

In [ ]:
# ============================================================
# CELL 2: Kaggle Download (with resume — skips if data already on Drive)
# ============================================================
#@title Enter Kaggle API Token { display-mode: "form" }
KAGGLE_TOKEN = 'YOUR_KAGGLE_API_TOKEN_HERE'  #@param {type:"string"}
import shutil

# Resume gate: skip download if data already exists
required_files = ['training_data_means.csv', 'training_data_ground_truth_table.csv',
                  'sample_submission.csv', 'training_cells.h5ad', 'pert_ids_val.csv']
existing_data = os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else []
all_present = all(f in existing_data for f in required_files)

if all_present:
    print(f'✓ RESUME: All {len(required_files)} required files already on Drive')
    print(f'  Files: {existing_data}')
    # Still need to check for pert_ids_all.csv
    if 'pert_ids_all.csv' not in existing_data:
        print('  ⚠ pert_ids_all.csv not found — will download')
    else:
        print('  ✓ pert_ids_all.csv present')
        KAGGLE_TOKEN = ''  # Skip download

if KAGGLE_TOKEN:
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'access_token'), 'w') as f: f.write(KAGGLE_TOKEN)
    os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
    !pip uninstall -y kaggle kagglehub kagglesdk 2>/dev/null
    !pip install -q --upgrade kaggle
    download_path = '/content/kaggle_data'
    os.makedirs(download_path, exist_ok=True)
    !kaggle competitions download -c echoes-of-silenced-genes -p {download_path}
    !unzip -q -o {download_path}/echoes-of-silenced-genes.zip -d {download_path}
    for f in os.listdir(download_path):
        src, dst = os.path.join(download_path, f), os.path.join(DATA_DIR, f)
        if os.path.isfile(src):
            if not os.path.exists(dst) or f == 'pert_ids_all.csv':
                shutil.copy2(src, dst)
    print(f'Data files: {os.listdir(DATA_DIR)}')
else:
    print(f'Using cached data: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "None"}')

In [ ]:
# ============================================================
# CELL 3: Dependencies (V_FINAL — streamlined, no GMM/NN needed)
# ============================================================
!pip -q install scanpy anndata

import os, gc, json, math, shutil, warnings, re, datetime, time, copy, zipfile
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.facecolor'] = 'white'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from sklearn.model_selection import KFold
from google.colab import files as colab_files
import requests

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
print('Dependencies ready (V_FINAL: Predict ALL 120 Perturbations)')

In [ ]:
# ============================================================
# CELL 4: Config — V_FINAL (exact V20 RUN=3 parameters, FROZEN)
#
# CRITICAL: Every parameter here is LOCKED to the V20 RUN=3
# configuration that achieved LB=4.08996 (all-time best).
# DO NOT CHANGE ANY VALUES.
# ============================================================
USE_AMP = True
USE_TF32 = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision('high')
    except: pass

@dataclass
class Config:
    # Data
    NUM_GENES: int = 5127
    NUM_GENES_FULL: int = 19226
    NUM_TRAIN_PERTS: int = 80
    NUM_ALL_PERTS: int = 120  # NEW: all 120 perturbations

    # PseudoBulk — 3-tier augmentation (LOCKED from V15/V20)
    N_RESAMPLES_FULL: int = 10
    N_RESAMPLES_HALF: int = 5
    HALF_CELL_FRACTION: float = 0.5
    MIN_CELLS_PER_PERT: int = 10
    MIN_CELLS_PER_CHANNEL: int = 5
    CTRL_SUBSAMPLE: int = 200
    PB_QUALITY_THRESHOLD: float = 0.3

    # LOO (LOCKED from V20 RUN=3 — baseline 5th percentile)
    LOO_PERCENTILE: float = 5.0
    LOO_MIN_CELLS: int = 10
    LOO_N_RESAMPLES: int = 3
    LOO_SAMPLE_WEIGHT: float = 0.05

    # Replogle augmentation (for Run B only)
    REPLOGLE_SAMPLE_WEIGHT: float = 0.03
    REPLOGLE_USE_ESSENTIAL: bool = True
    REPLOGLE_USE_GWPS: bool = True
    REPLOGLE_MIN_CELLS: int = 10
    REPLOGLE_GWPS_DISCOUNT: float = 0.75

    # MLP (V17 architecture — LOCKED)
    EMBED_DIM: int = 64
    MLP_DEPTH: int = 2
    MLP_HIDDEN: int = 384
    MLP_DROPOUT: float = 0.5
    MLP_EPOCHS: int = 300
    MLP_LR: float = 5e-4
    EARLY_STOP: int = 50
    WEIGHT_DECAY: float = 0.01

    # Loss (V16 tuned — LOCKED)
    LOSS: str = 'cosine_light'
    COSINE_LAMBDA: float = 0.08
    COS_RIGHT: float = 0.0405
    GT_UPWEIGHT: float = 4.5
    N_MLP_ENSEMBLE: int = 30

    # KO correction (V6 proven)
    KNOCKOUT_MULTIPLIER: float = 1.0

    # System
    N_FOLDS: int = 5
    SEED: int = 42
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.SEED)

def clean_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# === Checkpoint helpers ===
def save_stage(name, data):
    """Save a stage checkpoint to Drive."""
    path = os.path.join(VFINAL_DIR, f'{name}.npz')
    np.savez_compressed(path, **data)
    print(f'  ✓ Checkpoint saved: {name}')

def load_stage(name):
    """Load stage checkpoint if it exists."""
    path = os.path.join(VFINAL_DIR, f'{name}.npz')
    if os.path.exists(path):
        print(f'  ✓ Resumed: {name}')
        return dict(np.load(path, allow_pickle=True))
    return None

def save_json(name, data):
    path = os.path.join(VFINAL_DIR, f'{name}.json')
    with open(path, 'w') as f: json.dump(data, f, indent=2)

def load_json(name):
    path = os.path.join(VFINAL_DIR, f'{name}.json')
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return None

# === Run configurations ===
RUN_CONFIGS = {
    'A': {'name': 'V20_Baseline', 'replogle': False, 'desc': 'Pure V20 RUN=3 reproduction'},
    'B': {'name': 'V20_Replogle', 'replogle': True,  'desc': 'V20 + Replogle K562 augmentation'},
}

print(f'V_FINAL Config: device={cfg.DEVICE}')
print(f'  Architecture: V17 LightMLP H={cfg.MLP_HIDDEN}, D={cfg.MLP_DEPTH} (FROZEN)')
print(f'  Loss: cosine_light lambda={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT} (FROZEN)')
print(f'  LOO: Baseline {cfg.LOO_PERCENTILE}th pctile, {cfg.LOO_N_RESAMPLES} resamples, weight={cfg.LOO_SAMPLE_WEIGHT}')
print(f'  Ensemble: {cfg.N_MLP_ENSEMBLE} models')
print(f'  Prediction: ALL {cfg.NUM_ALL_PERTS} perturbations (60 public + 60 private)')
print(f'\nRun Configs:')
for rid, rc in RUN_CONFIGS.items():
    print(f'  Run {rid}: {rc["name"]} — {rc["desc"]}')

In [ ]:
# ============================================================
# CELL 5: Load Competition Data + pert_ids_all.csv (ALL 120)
#
# KEY CHANGE FROM V23: Loads pert_ids_all.csv instead of
# pert_ids_val.csv to get gene identities for ALL 120 perts.
# This enables KO correction for pert_61-120 (final ranking).
# ============================================================
print('=' * 70)
print('CELL 5: Load Competition Data + ALL 120 Perturbation IDs')
print('=' * 70)

data_path = DATA_DIR if os.path.exists(DATA_DIR) and os.listdir(DATA_DIR) else '/content/kaggle_data'
print(f'Data path: {data_path}')

# --- Training data ---
train_means = pd.read_csv(os.path.join(data_path, 'training_data_means.csv'))
first_col = train_means.iloc[:, 0].astype(str)
control_mask = first_col.str.lower().str.contains('non-targeting|control', na=False)
control_row = train_means[control_mask].iloc[0] if control_mask.any() else train_means.iloc[-1]
pert_df = train_means[~control_mask] if control_mask.any() else train_means.iloc[:-1]

gene_names = list(train_means.columns[1:])
control_expr = control_row.iloc[1:].values.astype(np.float32)
pert_names = pert_df.iloc[:, 0].tolist()
pert_expr = pert_df.iloc[:, 1:].values.astype(np.float32)
deltas = pert_expr - control_expr
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
pert_to_idx = {p: i for i, p in enumerate(pert_names)}
pert_gene_indices = np.array([gene_to_idx.get(p, -1) for p in pert_names])

# --- Ground truth weights ---
gt_df = pd.read_csv(os.path.join(data_path, 'training_data_ground_truth_table.csv'))
id_col = None
for c in ['pert_symbol', 'pert', 'gene', 'target_gene']:
    if c in gt_df.columns:
        id_col = c; break
if id_col:
    gt_df[id_col] = gt_df[id_col].astype(str)
    if set(pert_names).issubset(set(gt_df[id_col])):
        gt_df = gt_df.set_index(id_col).reindex(pert_names).reset_index()

weight_cols = [c for c in gt_df.columns if c.startswith('w_')]
weights_array = gt_df[weight_cols].values.astype(np.float32) if weight_cols else np.abs(deltas) + 0.1

# --- Sample submission (120 rows) ---
sample_submission = pd.read_csv(os.path.join(data_path, 'sample_submission.csv'))
SUBMISSION_COLUMNS = list(sample_submission.columns)
all_pert_ids = sample_submission['pert_id'].tolist()  # pert_1 through pert_120
assert len(all_pert_ids) == 120, f'Expected 120 pert_ids, got {len(all_pert_ids)}'

# --- CRITICAL: Load pert_ids_all.csv (NEW FILE with ALL 120 identities) ---
pert_all_path = os.path.join(data_path, 'pert_ids_all.csv')
assert os.path.exists(pert_all_path), f'pert_ids_all.csv not found! Download from Kaggle.'
pert_all_df = pd.read_csv(pert_all_path)
print(f'\n✓ Loaded pert_ids_all.csv: {len(pert_all_df)} perturbations')
print(f'  Columns: {list(pert_all_df.columns)}')

# Normalize pert_id
def _norm_pert_id(value):
    if value is None or (isinstance(value, float) and np.isnan(value)): return None
    s = str(value).strip()
    if not s: return None
    if s.startswith('pert_'): return s
    if s.isdigit(): return f'pert_{int(s)}'
    m = re.search(r'(\d+)', s)
    if m: return f'pert_{int(m.group(1))}'
    return s

pert_all_df['pert_id_norm'] = pert_all_df['pert_id'].apply(_norm_pert_id)

# Build mapping: pert_id → gene symbol for ALL 120 perturbations
pert_id_to_symbol_all = dict(zip(pert_all_df['pert_id_norm'], pert_all_df['pert']))

# Separate val (public) and test (private) sets
val_perts = pert_all_df[pert_all_df['class'] == 'val']
test_perts = pert_all_df[pert_all_df['class'] == 'test']
print(f'  Public (val):  {len(val_perts)} perturbations (pert_1 to pert_60)')
print(f'  Private (test): {len(test_perts)} perturbations (pert_61 to pert_120)')

# Build gene indices for ALL 120 perturbations
gene_names_upper = {g.upper(): g for g in gene_names}
all_gene_indices = []
all_pert_symbols = []
for pid in all_pert_ids:
    pid_norm = _norm_pert_id(pid)
    symbol = pert_id_to_symbol_all.get(pid_norm)
    all_pert_symbols.append(symbol)
    if symbol is not None and symbol in gene_to_idx:
        all_gene_indices.append(gene_to_idx[symbol])
    elif symbol is not None and symbol.upper() in gene_names_upper:
        all_gene_indices.append(gene_to_idx[gene_names_upper[symbol.upper()]])
    else:
        all_gene_indices.append(-1)

valid_public = sum(1 for i in all_gene_indices[:60] if i >= 0)
valid_private = sum(1 for i in all_gene_indices[60:] if i >= 0)
print(f'\nGene index resolution:')
print(f'  Public  (pert_1-60):   {valid_public}/60 resolved')
print(f'  Private (pert_61-120): {valid_private}/60 resolved')
print(f'  Total:                 {valid_public + valid_private}/120 resolved')

# --- Diagnostic: Show test perturbation categories ---
print(f'\n--- Final Test Perturbations (pert_61-120) ---')
test_genes = [all_pert_symbols[i] for i in range(60, 120)]
test_in_5127 = sum(1 for g in test_genes if g and (g in gene_to_idx or g.upper() in gene_names_upper))
test_in_train = sum(1 for g in test_genes if g and g in pert_to_idx)
print(f'  Test genes in 5127-gene space: {test_in_5127}/60')
print(f'  Test genes overlapping with training perts: {test_in_train}/60')

# Show first 10 test genes and their resolution
print(f'\n  Sample test gene resolutions:')
for i in range(60, min(70, 120)):
    symbol = all_pert_symbols[i]
    gi = all_gene_indices[i]
    status = f'idx={gi}' if gi >= 0 else 'NOT FOUND'
    print(f'    {all_pert_ids[i]}: {symbol} → {status}')

# --- Summary chart ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('V_FINAL: ALL 120 Perturbation Analysis', fontsize=14, fontweight='bold')

# Chart 1: Gene resolution status
ax = axes[0]
categories = ['Public\n(pert_1-60)', 'Private\n(pert_61-120)']
resolved = [valid_public, valid_private]
unresolved = [60 - valid_public, 60 - valid_private]
x = np.arange(len(categories))
ax.bar(x, resolved, color='#4CAF50', label='Resolved')
ax.bar(x, unresolved, bottom=resolved, color='#F44336', label='Unresolved')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Count')
ax.set_title('Gene Index Resolution (for KO correction)')
ax.legend()
for i, (r, u) in enumerate(zip(resolved, unresolved)):
    ax.text(i, r/2, str(r), ha='center', va='center', fontweight='bold', color='white', fontsize=14)

# Chart 2: Delta magnitude distribution (training)
ax = axes[1]
abs_deltas = np.abs(deltas).mean(axis=1)
ax.hist(abs_deltas, bins=30, color='#2196F3', alpha=0.8, edgecolor='white')
ax.axvline(abs_deltas.mean(), color='red', linestyle='--', label=f'Mean={abs_deltas.mean():.4f}')
ax.set_title(f'Training Delta Magnitudes (n={len(pert_names)})')
ax.set_xlabel('Mean |Delta|')
ax.set_ylabel('Count')
ax.legend()

# Chart 3: Data summary
ax = axes[2]
ax.axis('off')
summary = (
    f'V_FINAL Data Summary\n'
    f'{"─"*45}\n'
    f'Competition genes:    {len(gene_names):,}\n'
    f'Training perts:       {len(pert_names)}\n'
    f'ALL perturbations:    {len(all_pert_ids)}\n'
    f'  Public (val):       60\n'
    f'  Private (test):     60\n'
    f'{"─"*45}\n'
    f'Gene resolution:\n'
    f'  Public:  {valid_public}/60\n'
    f'  Private: {valid_private}/60\n'
    f'{"─"*45}\n'
    f'Deltas: {deltas.shape}\n'
    f'  Range: [{deltas.min():.3f}, {deltas.max():.3f}]\n'
    f'  Mean |delta|: {abs_deltas.mean():.4f}\n'
    f'Weight cols: {len(weight_cols)}\n'
)
ax.text(0.02, 0.95, summary, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()
print('✓ Data loaded with ALL 120 perturbation identities')

In [ ]:
# ============================================================
# CELL 6: Competition Metric + KO Stats (from V23 — unchanged)
# ============================================================
print('=' * 70)
print('CELL 6: Competition Metric + Knockout Statistics')
print('=' * 70)

baseline_pred = np.mean(deltas, axis=0)

baseline_wmae = None
if 'baseline_wmae' in gt_df.columns:
    baseline_wmae = gt_df['baseline_wmae'].values.astype(np.float64)
if baseline_wmae is None:
    baseline_wmae = np.mean(np.abs(deltas - baseline_pred) * weights_array, axis=1)

class CompetitionMetric:
    def __init__(self, eps=1e-12, max_log2=5.0, left=0.0, right=0.2):
        self.eps, self.max_log2, self.left, self.right = eps, max_log2, left, right

    def smoothstep(self, t):
        return t * t * (3.0 - 2.0 * t)

    def gate_smoothstep(self, x):
        t = np.clip((x - self.left) / (self.right - self.left), 0.0, 1.0)
        return self.smoothstep(t)

    def weighted_cosine(self, a, b):
        a, b = np.asarray(a, dtype=np.float64).ravel(), np.asarray(b, dtype=np.float64).ravel()
        x = np.maximum(np.abs(a), np.abs(b))
        w2 = self.gate_smoothstep(x) ** 2
        num = np.sum(w2 * a * b)
        den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
        return float(num / den) if den > self.eps else 0.0

    def score(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        if base.shape[0] != y_true.shape[0]:
            base = np.mean(np.abs(y_true - baseline_pred) * w, axis=1)
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        sum_wmae = float(np.sum(terms))
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return round(float(sum_wmae * max(0.0, wcos)), 5), sum_wmae, wcos

    def score_per_pert(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return terms, wcos

metric = CompetitionMetric()

# KO effect statistics from training data
ko_effects = [deltas[i, idx] for i, idx in enumerate(pert_gene_indices) if idx >= 0]
knockout_stats = {'mean': np.mean(ko_effects), 'std': np.std(ko_effects),
                  'median': np.median(ko_effects), 'min': np.min(ko_effects), 'max': np.max(ko_effects)}

def apply_ko(pred, idx, stats, mult=1.0):
    pred = pred.copy()
    if 0 <= idx < len(pred):
        pred[idx] = stats['mean'] * mult
    return pred

# KO stats chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Knockout Effect Analysis', fontsize=13, fontweight='bold')

ax = axes[0]
ax.hist(ko_effects, bins=30, color='#E91E63', alpha=0.8, edgecolor='white')
ax.axvline(knockout_stats['mean'], color='black', linestyle='--', linewidth=2,
           label=f'Mean={knockout_stats["mean"]:.4f}')
ax.axvline(knockout_stats['median'], color='blue', linestyle=':', linewidth=2,
           label=f'Median={knockout_stats["median"]:.4f}')
ax.set_title(f'Self-Targeting KO Delta Distribution (n={len(ko_effects)})')
ax.set_xlabel('KO Gene Delta')
ax.legend()

ax = axes[1]
ax.axis('off')
ko_text = (
    f'Knockout Statistics\n'
    f'{"─"*35}\n'
    f'Mean:   {knockout_stats["mean"]:+.4f}\n'
    f'Std:    {knockout_stats["std"]:.4f}\n'
    f'Median: {knockout_stats["median"]:+.4f}\n'
    f'Min:    {knockout_stats["min"]:+.4f}\n'
    f'Max:    {knockout_stats["max"]:+.4f}\n'
    f'{"─"*35}\n'
    f'KO correction: pred[gene] = mean * {cfg.KNOCKOUT_MULTIPLIER}\n'
    f'Applied to ALL 120 perturbations\n'
    f'  (60 public + 60 private)\n'
    f'\nBaseline WMAE: mean={baseline_wmae.mean():.4f}\n'
)
ax.text(0.1, 0.95, ko_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Metric ready. KO mean={knockout_stats["mean"]:.4f}')

In [ ]:
# ============================================================
# CELL 7: 3-Tier PseudoBulk Generation (from V23 — unchanged)
#
# Resume: Checks for cached PB data on Drive before regenerating.
# Tier 1: Full bootstrap (10 resamples per pert, all cells)
# Tier 2: Per-channel means (1 sample per pert×channel)
# Tier 3: Half-cell bootstrap (5 resamples, 50% cells)
# Quality gate: corr > 0.3 with GT delta
# Caches ctrl_dense for LOO generation in Cell 8
# ============================================================
import scanpy as sc

print('=' * 70)
print('CELL 7: 3-Tier PseudoBulk Generation')
print('=' * 70)

# === Resume gate ===
PB_CACHE = os.path.join(VFINAL_DIR, 'pb_data.npz')
CTRL_CACHE = os.path.join(VFINAL_DIR, 'ctrl_dense.npz')

if os.path.exists(PB_CACHE) and os.path.exists(CTRL_CACHE):
    print('✓ RESUME: Loading cached PseudoBulk + control data...')
    pb_data = np.load(PB_CACHE, allow_pickle=True)
    pb_deltas = pb_data['deltas']
    pb_pert_indices = pb_data['pert_indices']
    pb_pert_names = list(pb_data['pert_names'])
    pb_cell_weights = pb_data['cell_weights']
    pb_weights = np.zeros((len(pb_deltas), cfg.NUM_GENES), dtype=np.float32)
    for i, pn in enumerate(pb_pert_names):
        pb_weights[i] = weights_array[pert_to_idx[pn]]

    ctrl_data = np.load(CTRL_CACHE)
    ctrl_dense = ctrl_data['ctrl_dense']
    comp_gene_order = ctrl_data['comp_gene_order']

    print(f'  PB: {len(pb_deltas)} samples, {len(set(pb_pert_names))} perts')
    print(f'  Control: {ctrl_dense.shape[0]} cells, {ctrl_dense.shape[1]} genes')
else:
    print('Generating PseudoBulk from scratch...')
    h5ad_path = os.path.join(data_path, 'training_cells.h5ad')
    assert os.path.exists(h5ad_path), f'training_cells.h5ad not found at {h5ad_path}'

    adata_train = sc.read_h5ad(h5ad_path)
    print(f'Shape: {adata_train.shape[0]} cells x {adata_train.shape[1]} genes')

    X_check = adata_train.X[:200]
    if sp.issparse(X_check): X_check = X_check.toarray()
    data_max = X_check.max()
    IS_RAW = data_max > 50
    print(f'Data max: {data_max:.1f} -> {"Raw UMI" if IS_RAW else "Already normalized"}')

    h5_genes = list(adata_train.var_names)
    h5_gene_upper = {g.upper(): i for i, g in enumerate(h5_genes)}
    gene_subset_idx = []
    comp_gene_order = []
    for ci, g in enumerate(gene_names):
        gu = g.upper()
        if gu in h5_gene_upper:
            gene_subset_idx.append(h5_gene_upper[gu])
            comp_gene_order.append(ci)
    gene_subset_idx = np.array(gene_subset_idx)
    comp_gene_order = np.array(comp_gene_order)
    print(f'Gene overlap: {len(gene_subset_idx)}/{cfg.NUM_GENES}')

    # Find perturbation column
    CANDIDATE_COLS = ['pert_symbol', 'perturbation', 'pert', 'gene', 'target_gene',
                      'perturbation_name', 'gene_symbol', 'gene_name', 'symbol']
    def _count_pert_matches(col_series, pert_names_list):
        col_upper = set(col_series.astype(str).str.upper().unique())
        return sum(1 for p in pert_names_list if p.upper() in col_upper)

    pert_col_h5 = None; pert_col_matches = 0; _sgrna_to_gene = {}
    for c in CANDIDATE_COLS:
        if c in adata_train.obs.columns:
            n_matches = _count_pert_matches(adata_train.obs[c], pert_names)
            if n_matches > pert_col_matches:
                pert_col_h5, pert_col_matches = c, n_matches

    if pert_col_matches < len(pert_names) // 2:
        for c in adata_train.obs.columns:
            if c == pert_col_h5: continue
            col = adata_train.obs[c]
            if col.dtype == object or str(col.dtype) == 'category':
                if 5 < col.nunique() < 50000:
                    n_matches = _count_pert_matches(col, pert_names)
                    if n_matches > pert_col_matches:
                        pert_col_h5, pert_col_matches = c, n_matches

    if pert_col_matches < len(pert_names) // 2:
        pert_names_upper = {p.upper(): p for p in pert_names}
        for c in adata_train.obs.columns:
            col = adata_train.obs[c]
            if col.dtype != object and str(col.dtype) != 'category': continue
            extraction_map = {}
            for val in col.astype(str).unique():
                for delim in ['_', '-', '.', '|', ':']:
                    for part in val.upper().split(delim):
                        part = part.strip()
                        if part in pert_names_upper:
                            extraction_map[val] = pert_names_upper[part]; break
                    if val in extraction_map: break
            if len(extraction_map) > pert_col_matches:
                pert_col_h5, pert_col_matches = c, len(extraction_map)
                _sgrna_to_gene = extraction_map

    assert pert_col_h5 is not None
    print(f'Perturbation column: "{pert_col_h5}" ({pert_col_matches}/{len(pert_names)} matches)')

    pert_str_h5 = adata_train.obs[pert_col_h5].astype(str)
    ctrl_mask_h5 = pert_str_h5.str.lower().str.contains('non.targeting|control|ctrl', regex=True, na=False)
    if 'is_control' in adata_train.obs.columns:
        ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['is_control'].astype(bool)
    if 'control' in adata_train.obs.columns and adata_train.obs['control'].dtype == bool:
        ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['control']
    ctrl_cell_idx = np.where(ctrl_mask_h5.values)[0]
    print(f'Control cells: {len(ctrl_cell_idx)}')

    channel_col = None
    for c in ['channel', 'batch', 'lane', 'pool', 'sample']:
        if c in adata_train.obs.columns:
            n_unique = adata_train.obs[c].nunique()
            if 2 <= n_unique <= 20: channel_col = c; break
    if channel_col is None:
        for c in adata_train.obs.columns:
            col = adata_train.obs[c]
            if col.dtype == object or str(col.dtype) == 'category':
                if 2 <= col.nunique() <= 20 and c != pert_col_h5:
                    channel_col = c; break
    channel_names = sorted(adata_train.obs[channel_col].astype(str).unique()) if channel_col else []
    if channel_col: print(f'Channel column: "{channel_col}" ({len(channel_names)} channels)')

    # Build cell indices per perturbation
    cells_per_pert = {}
    if _sgrna_to_gene:
        gene_to_sgrna_ids = {}
        for sgrna_val, gene_name in _sgrna_to_gene.items():
            gene_to_sgrna_ids.setdefault(gene_name, []).append(sgrna_val)
        for p in pert_names:
            if p in gene_to_sgrna_ids:
                all_idx = []
                for sgrna_val in gene_to_sgrna_ids[p]:
                    mask = (pert_str_h5 == sgrna_val) & (~ctrl_mask_h5)
                    all_idx.extend(np.where(mask.values)[0].tolist())
                if len(all_idx) >= 3: cells_per_pert[p] = np.array(all_idx)
    for p in pert_names:
        if p in cells_per_pert: continue
        mask = (pert_str_h5.str.upper() == p.upper()) & (~ctrl_mask_h5)
        cell_idx = np.where(mask.values)[0]
        if len(cell_idx) >= 3: cells_per_pert[p] = cell_idx
    print(f'Perturbations with cells: {len(cells_per_pert)}/{len(pert_names)}')

    # Per-channel cells
    cells_per_pert_channel = {}; ctrl_cells_per_channel = {}
    if channel_col:
        obs_ch = adata_train.obs[channel_col].astype(str).values
        for ch in channel_names:
            ch_mask = obs_ch == ch
            ctrl_ch_idx = np.where(ch_mask & ctrl_mask_h5.values)[0]
            if len(ctrl_ch_idx) > 0: ctrl_cells_per_channel[ch] = ctrl_ch_idx
        for p, cell_idx in cells_per_pert.items():
            for ch in channel_names:
                ch_mask_p = obs_ch[cell_idx] == ch
                ch_idx = cell_idx[ch_mask_p]
                if len(ch_idx) >= cfg.MIN_CELLS_PER_CHANNEL:
                    cells_per_pert_channel[(p, ch)] = ch_idx

    # Normalize and cache
    print(f'\nNormalizing and caching...')
    t0_cache = time.time()
    def normalize_cells_to_dense(X_raw, gene_subset_idx, is_raw=True):
        if sp.issparse(X_raw): X_raw = X_raw.toarray()
        X_raw = X_raw.astype(np.float64)
        if is_raw:
            totals = X_raw.sum(axis=1, keepdims=True); totals[totals == 0] = 1
            normalized_full = np.log2(1.0 + X_raw / totals * 10000.0)
        else: normalized_full = X_raw
        return normalized_full[:, gene_subset_idx].astype(np.float32)

    ctrl_dense = normalize_cells_to_dense(adata_train.X[ctrl_cell_idx], gene_subset_idx, is_raw=IS_RAW)
    ctrl_dense_per_channel = {}
    if channel_col:
        for ch, ch_idx in ctrl_cells_per_channel.items():
            ctrl_dense_per_channel[ch] = normalize_cells_to_dense(adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

    pert_cells_dense = {}
    for p, cell_idx in tqdm(cells_per_pert.items(), desc='Caching pert cells'):
        pert_cells_dense[p] = normalize_cells_to_dense(adata_train.X[cell_idx], gene_subset_idx, is_raw=IS_RAW)

    pert_cells_dense_channel = {}
    if channel_col:
        for (p, ch), ch_idx in tqdm(cells_per_pert_channel.items(), desc='Caching pert-channel'):
            pert_cells_dense_channel[(p, ch)] = normalize_cells_to_dense(adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

    print(f'  Caching done in {time.time()-t0_cache:.1f}s')
    del adata_train; clean_memory()

    # Save ctrl_dense to Drive for resume
    np.savez_compressed(CTRL_CACHE, ctrl_dense=ctrl_dense, comp_gene_order=comp_gene_order)
    print(f'  ✓ ctrl_dense saved to Drive ({ctrl_dense.shape})')

    # Generate 3-Tier PseudoBulk
    print(f'\nGenerating 3-Tier PseudoBulk...')
    rng = np.random.RandomState(cfg.SEED)
    n_ctrl = ctrl_dense.shape[0]; n_ctrl_sub = min(n_ctrl, cfg.CTRL_SUBSAMPLE)
    pb_deltas_list, pb_pert_indices_list, pb_pert_names_list = [], [], []
    pb_tiers_list, pb_n_cells_list = [], []

    # Tier 1: Full Bootstrap
    n_t1 = 0
    for p in pert_names:
        if p not in pert_cells_dense: continue
        pert_cells = pert_cells_dense[p]; n_pert = pert_cells.shape[0]
        if n_pert < cfg.MIN_CELLS_PER_PERT: continue
        gi = pert_gene_indices[pert_to_idx[p]]
        for _ in range(cfg.N_RESAMPLES_FULL):
            pert_idx = rng.randint(0, n_pert, size=n_pert)
            ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
            pb_deltas_list.append(delta); pb_pert_indices_list.append(gi)
            pb_pert_names_list.append(p); pb_tiers_list.append(1); pb_n_cells_list.append(n_pert); n_t1 += 1
    print(f'  Tier 1: {n_t1}')

    # Tier 2: Per-Channel
    n_t2 = 0
    if channel_col and ctrl_dense_per_channel:
        for p in pert_names:
            gi = pert_gene_indices[pert_to_idx[p]]
            for ch in channel_names:
                if (p, ch) not in pert_cells_dense_channel or ch not in ctrl_dense_per_channel: continue
                pert_ch = pert_cells_dense_channel[(p, ch)]; ctrl_ch = ctrl_dense_per_channel[ch]
                delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
                delta[comp_gene_order] = pert_ch.mean(axis=0) - ctrl_ch.mean(axis=0)
                pb_deltas_list.append(delta); pb_pert_indices_list.append(gi)
                pb_pert_names_list.append(p); pb_tiers_list.append(2); pb_n_cells_list.append(pert_ch.shape[0]); n_t2 += 1
    print(f'  Tier 2: {n_t2}')

    # Tier 3: Half-Cell
    n_t3 = 0
    for p in pert_names:
        if p not in pert_cells_dense: continue
        pert_cells = pert_cells_dense[p]; n_pert = pert_cells.shape[0]
        if n_pert < cfg.MIN_CELLS_PER_PERT: continue
        gi = pert_gene_indices[pert_to_idx[p]]
        half_n = max(3, int(n_pert * cfg.HALF_CELL_FRACTION))
        for _ in range(cfg.N_RESAMPLES_HALF):
            pert_idx = rng.choice(n_pert, size=half_n, replace=False)
            ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
            pb_deltas_list.append(delta); pb_pert_indices_list.append(gi)
            pb_pert_names_list.append(p); pb_tiers_list.append(3); pb_n_cells_list.append(half_n); n_t3 += 1
    print(f'  Tier 3: {n_t3}')

    # Quality Gate
    pb_deltas_raw = np.array(pb_deltas_list, dtype=np.float32)
    pb_pert_indices_raw = np.array(pb_pert_indices_list)
    pb_tiers_raw = np.array(pb_tiers_list)
    pb_n_cells_raw = np.array(pb_n_cells_list)
    keep_mask = np.ones(len(pb_deltas_raw), dtype=bool)
    for i in range(len(pb_deltas_raw)):
        pn = pb_pert_names_list[i]
        corr = np.corrcoef(pb_deltas_raw[i], deltas[pert_to_idx[pn]])[0, 1]
        if np.isnan(corr) or corr < cfg.PB_QUALITY_THRESHOLD: keep_mask[i] = False
    print(f'  Quality gate: kept {keep_mask.sum()}/{len(pb_deltas_raw)}')

    pb_deltas = pb_deltas_raw[keep_mask]
    pb_pert_indices = pb_pert_indices_raw[keep_mask]
    pb_pert_names = [pb_pert_names_list[i] for i in range(len(pb_pert_names_list)) if keep_mask[i]]
    pb_n_cells = pb_n_cells_raw[keep_mask]
    pb_cell_weights = np.sqrt(pb_n_cells.astype(np.float32))
    mean_sqrt = pb_cell_weights.mean()
    if mean_sqrt > 0: pb_cell_weights /= mean_sqrt

    pb_weights = np.zeros((len(pb_deltas), cfg.NUM_GENES), dtype=np.float32)
    for i, pn in enumerate(pb_pert_names):
        pb_weights[i] = weights_array[pert_to_idx[pn]]

    # Save PB to Drive for resume
    np.savez_compressed(PB_CACHE,
                        deltas=pb_deltas, pert_indices=pb_pert_indices,
                        pert_names=np.array(pb_pert_names, dtype=object),
                        cell_weights=pb_cell_weights)
    print(f'  ✓ PB data saved to Drive')

    del pert_cells_dense, pert_cells_dense_channel; clean_memory()

# Summary chart
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.bar(['PB Samples', 'GT Samples', 'Total'],
       [len(pb_deltas), len(pert_names), len(pb_deltas) + len(pert_names)],
       color=['#2196F3', '#E91E63', '#4CAF50'], alpha=0.85, edgecolor='white')
for i, v in enumerate([len(pb_deltas), len(pert_names), len(pb_deltas) + len(pert_names)]):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
ax.set_title('PseudoBulk Data Ready')
ax.set_ylabel('Sample Count')
plt.tight_layout()
plt.show()

n_pb_perts = len(set(pb_pert_names))
print(f'\nPB ready: {n_pb_perts} perts, {len(pb_deltas)} samples')
print(f'Total so far: {len(pb_deltas)} PB + {len(pert_names)} GT = {len(pb_deltas) + len(pert_names)}')

In [ ]:
# ============================================================
# CELL 8: LOO Baseline Generation (V20 RUN=3 method ONLY)
#
# Resume: Tries V23 cache first, then VFINAL cache, then generates.
# Only uses BASELINE 5th percentile — no NN-matched/adaptive/etc.
# This is the proven V20 RUN=3 LOO that achieved LB=4.090.
# ============================================================
print('=' * 70)
print('CELL 8: LOO Baseline Synthetic Perturbations')
print('=' * 70)

# === Resume: check for existing LOO cache ===
LOO_CACHE_VFINAL = os.path.join(VFINAL_DIR, 'loo_baseline.npz')
LOO_CACHE_V23 = os.path.join(V23_DIR, 'loo_baseline.npz') if os.path.exists(V23_DIR) else None

loo_loaded = False

# Try VFINAL cache first
if os.path.exists(LOO_CACHE_VFINAL):
    print('✓ RESUME: Loading LOO from V_FINAL cache...')
    loo_data = np.load(LOO_CACHE_VFINAL)
    loo_gene_indices = loo_data['gis']
    loo_deltas_all = loo_data['deltas']
    loo_n_genes = int(loo_data['n_genes'])
    loo_loaded = True

# Try V23 cache
elif LOO_CACHE_V23 and os.path.exists(LOO_CACHE_V23):
    print('✓ RESUME: Loading LOO from V23 cache (reusing)...')
    loo_data = np.load(LOO_CACHE_V23)
    loo_gene_indices = loo_data['gis']
    loo_deltas_all = loo_data['deltas']
    loo_n_genes = int(loo_data['n_genes'])
    # Copy to VFINAL cache
    np.savez_compressed(LOO_CACHE_VFINAL, gis=loo_gene_indices,
                        deltas=loo_deltas_all, n_genes=loo_n_genes)
    print(f'  ✓ Copied to V_FINAL cache')
    loo_loaded = True

if loo_loaded:
    print(f'  LOO baseline: {loo_n_genes} genes, {len(loo_gene_indices)} samples')
else:
    print('Generating LOO baseline from scratch...')
    # Need ctrl_dense and comp_gene_order from Cell 7
    assert 'ctrl_dense' in dir(), 'ctrl_dense not found — re-run Cell 7'

    comp_gene_to_col = {}
    for col_j, comp_gi in enumerate(comp_gene_order):
        comp_gene_to_col[comp_gi] = col_j
    n_ctrl_cells = ctrl_dense.shape[0]
    percentile_threshold = cfg.LOO_PERCENTILE

    print(f'Control cells: {n_ctrl_cells}')
    print(f'LOO resamples: {cfg.LOO_N_RESAMPLES}, Percentile: {percentile_threshold}%')

    t0 = time.time()
    rng = np.random.RandomState(cfg.SEED)
    loo_deltas_list, loo_gis_list = [], []
    n_genes_done = 0

    for comp_gi in tqdm(range(cfg.NUM_GENES), desc='LOO baseline'):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]
        threshold = np.percentile(gene_expr, percentile_threshold)
        low_mask = gene_expr <= threshold
        n_low = low_mask.sum()
        if n_low < cfg.LOO_MIN_CELLS: continue
        low_cells = ctrl_dense[low_mask]
        for _ in range(cfg.LOO_N_RESAMPLES):
            boot_idx = rng.randint(0, n_low, size=n_low)
            ctrl_idx = rng.randint(0, n_ctrl_cells, size=min(n_ctrl_cells, cfg.CTRL_SUBSAMPLE))
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = low_cells[boot_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
            loo_deltas_list.append(delta)
            loo_gis_list.append(comp_gi)
        n_genes_done += 1

    loo_gene_indices = np.array(loo_gis_list, dtype=np.int64)
    loo_deltas_all = np.array(loo_deltas_list, dtype=np.float32)
    loo_n_genes = n_genes_done
    elapsed = time.time() - t0

    print(f'  Baseline: {loo_n_genes} genes → {len(loo_gene_indices)} samples in {elapsed:.1f}s')

    # Save to Drive
    np.savez_compressed(LOO_CACHE_VFINAL, gis=loo_gene_indices,
                        deltas=loo_deltas_all, n_genes=loo_n_genes)
    print(f'  ✓ Cached to {LOO_CACHE_VFINAL}')

# Build LOO sample weights
loo_sample_weights_all = np.full(len(loo_gene_indices), cfg.LOO_SAMPLE_WEIGHT, dtype=np.float32)

# === Diagnostic Charts ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LOO Baseline Diagnostics (V20 RUN=3 Method)', fontsize=13, fontweight='bold')

# Chart 1: Sample count
ax = axes[0]
unique_genes = len(set(loo_gene_indices.tolist()))
ax.bar(['Genes\nCovered', 'Total\nSamples', 'Samples\nPer Gene'],
       [unique_genes, len(loo_gene_indices), len(loo_gene_indices) / max(unique_genes, 1)],
       color=['#2196F3', '#4CAF50', '#FF9800'], alpha=0.85, edgecolor='white')
for i, v in enumerate([unique_genes, len(loo_gene_indices), len(loo_gene_indices) / max(unique_genes, 1)]):
    ax.text(i, v + max(v*0.02, 10), f'{v:.0f}', ha='center', fontweight='bold')
ax.set_title('LOO Coverage')

# Chart 2: Delta magnitude distribution
ax = axes[1]
loo_abs = np.abs(loo_deltas_all).mean(axis=1)
gt_abs = np.abs(deltas).mean(axis=1)
ax.hist(loo_abs, bins=60, alpha=0.6, color='#2196F3', label=f'LOO (n={len(loo_deltas_all):,})', density=True)
ax.hist(gt_abs, bins=30, alpha=0.6, color='#E91E63', label=f'GT (n={len(pert_names)})', density=True)
ax.set_title('Delta Magnitude: LOO vs Ground Truth')
ax.set_xlabel('Mean |Delta|')
ax.legend(fontsize=9)

# Chart 3: KO self-targeting delta
ax = axes[2]
ko_deltas_loo = [loo_deltas_all[i, loo_gene_indices[i]] for i in range(len(loo_gene_indices))
                 if loo_gene_indices[i] < cfg.NUM_GENES]
gt_ko = [deltas[i, pert_gene_indices[i]] for i in range(len(pert_names)) if pert_gene_indices[i] >= 0]
ax.hist(ko_deltas_loo, bins=60, alpha=0.6, color='#2196F3', label=f'LOO KO ({len(ko_deltas_loo):,})', density=True)
ax.hist(gt_ko, bins=30, alpha=0.6, color='#E91E63', label=f'GT KO ({len(gt_ko)})', density=True)
ax.axvline(knockout_stats['mean'], color='red', linestyle='--', linewidth=2,
           label=f'GT KO mean={knockout_stats["mean"]:.3f}')
ax.set_title('Self-Targeting (KO) Delta')
ax.set_xlabel('KO Gene Delta')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# Check coverage of test perturbation genes
test_gene_indices_set = set(all_gene_indices[60:])
loo_gene_set = set(loo_gene_indices.tolist())
test_covered = len(test_gene_indices_set & loo_gene_set)
print(f'\nLOO coverage of FINAL test genes: {test_covered}/{len(test_gene_indices_set)}')
print(f'LOO ready: {loo_n_genes} genes, {len(loo_gene_indices):,} samples')

In [ ]:
# ============================================================
# CELL 9: Replogle K562 Download & Processing (for Run B)
#
# Resume: Tries V23 cache first, then VFINAL cache, then downloads.
# RAM-SAFE: files > 3 GB use backed (memory-mapped) mode
# with single-pass streaming — never loads full matrix into RAM.
# Maps Replogle genes → competition 5127-gene space.
# ============================================================
print('=' * 70)
print('CELL 9: Replogle K562 Data Augmentation (for Run B)')
print('=' * 70)

REPLOGLE_CACHE_VFINAL = os.path.join(VFINAL_DIR, 'replogle_augmentation.npz')
REPLOGLE_CACHE_V23 = os.path.join(V23_DIR, 'replogle_augmentation.npz') if os.path.exists(V23_DIR) else None

replogle_loaded = False

# Try VFINAL cache
if os.path.exists(REPLOGLE_CACHE_VFINAL):
    print('✓ RESUME: Loading Replogle from V_FINAL cache...')
    rdata = np.load(REPLOGLE_CACHE_VFINAL, allow_pickle=True)
    replogle_deltas = rdata['deltas']
    replogle_gene_indices = rdata['gene_indices']
    replogle_names = list(rdata['names'])
    replogle_loaded = True

# Try V23 cache
elif REPLOGLE_CACHE_V23 and os.path.exists(REPLOGLE_CACHE_V23):
    print('✓ RESUME: Loading Replogle from V23 cache (reusing)...')
    rdata = np.load(REPLOGLE_CACHE_V23, allow_pickle=True)
    replogle_deltas = rdata['deltas']
    replogle_gene_indices = rdata['gene_indices']
    replogle_names = list(rdata['names'])
    # Copy to VFINAL
    np.savez_compressed(REPLOGLE_CACHE_VFINAL, deltas=replogle_deltas,
                        gene_indices=replogle_gene_indices,
                        names=np.array(replogle_names, dtype=object))
    print(f'  ✓ Copied to V_FINAL cache')
    replogle_loaded = True

if replogle_loaded:
    n_replogle = len(replogle_deltas)
    print(f'  Replogle: {n_replogle} perturbations loaded')
else:
    print('Downloading and processing Replogle data from scratch...')
    import scanpy as sc

    SCPERTURB_URLS = {
        'ReplogleK562essential': 'https://zenodo.org/records/13350497/files/ReplogleWeissman2022_K562_essential.h5ad?download=1',
        'ReplogleK562gwps': 'https://zenodo.org/records/13350497/files/ReplogleWeissman2022_K562_gwps.h5ad?download=1',
    }

    def download_h5ad(name, url, dest_dir='/content'):
        fname = f'{name}.h5ad'
        fpath = os.path.join(dest_dir, fname)
        if os.path.exists(fpath):
            print(f'  {name}: already downloaded ({os.path.getsize(fpath)/1e9:.1f} GB)')
            return fpath
        print(f'  Downloading {name}...')
        r = requests.get(url, stream=True)
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(fpath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192*1024):
                f.write(chunk); downloaded += len(chunk)
                if total > 0:
                    print(f'\r  {name}: {downloaded/1e9:.2f}/{total/1e9:.2f} GB ({100*downloaded/total:.0f}%)', end='', flush=True)
        print(f'\n  ✓ {name} downloaded ({os.path.getsize(fpath)/1e9:.1f} GB)')
        return fpath

    def _normalize_chunk(X_chunk, is_raw):
        X_chunk = X_chunk.astype(np.float64)
        if is_raw:
            totals = X_chunk.sum(axis=1, keepdims=True); totals[totals == 0] = 1
            return np.log2(1.0 + X_chunk / totals * 10000.0)
        return X_chunk

    def _chunked_mean(X_sparse, row_indices, is_raw, chunk_size=5000):
        n_rows = len(row_indices); n_genes = X_sparse.shape[1]
        running_sum = np.zeros(n_genes, dtype=np.float64)
        for start in range(0, n_rows, chunk_size):
            end = min(start + chunk_size, n_rows)
            idx_chunk = row_indices[start:end]
            X_chunk = X_sparse[idx_chunk]
            if sp.issparse(X_chunk): X_chunk = X_chunk.toarray()
            running_sum += _normalize_chunk(X_chunk, is_raw).sum(axis=0)
            del X_chunk
        return (running_sum / n_rows).astype(np.float32)

    def _stream_perturbation_deltas(adata_backed, pert_labels, ctrl_mask, is_raw, min_cells, chunk_size=8000):
        n_cells = adata_backed.shape[0]; n_genes = adata_backed.shape[1]
        unique_perts = sorted(set(p for p in pert_labels.unique() if not ctrl_mask[pert_labels == p].all()))
        pert_sums = {p: np.zeros(n_genes, dtype=np.float64) for p in unique_perts}
        pert_counts = {p: 0 for p in unique_perts}
        ctrl_sum = np.zeros(n_genes, dtype=np.float64); ctrl_count = 0
        labels_arr = pert_labels.values.astype(str); ctrl_arr = ctrl_mask.values; pert_set = set(unique_perts)
        n_chunks = (n_cells + chunk_size - 1) // chunk_size
        for ci in tqdm(range(n_chunks), desc='  Streaming chunks'):
            start = ci * chunk_size; end = min(start + chunk_size, n_cells)
            X_chunk = adata_backed.X[start:end]
            if sp.issparse(X_chunk): X_chunk = X_chunk.toarray()
            X_norm = _normalize_chunk(X_chunk, is_raw); del X_chunk
            chunk_ctrl = ctrl_arr[start:end]; chunk_labels = labels_arr[start:end]
            if chunk_ctrl.any():
                ctrl_sum += X_norm[chunk_ctrl].sum(axis=0); ctrl_count += int(chunk_ctrl.sum())
            non_ctrl = ~chunk_ctrl
            if non_ctrl.any():
                nc_labels = chunk_labels[non_ctrl]; nc_X = X_norm[non_ctrl]
                for p in np.unique(nc_labels):
                    if p not in pert_set: continue
                    pmask = nc_labels == p
                    pert_sums[p] += nc_X[pmask].sum(axis=0); pert_counts[p] += int(pmask.sum())
                del nc_labels, nc_X
            del X_norm
            if ci % 20 == 0: gc.collect()
        ctrl_mean = (ctrl_sum / max(ctrl_count, 1)).astype(np.float32)
        deltas_dict = {}
        for p in unique_perts:
            if pert_counts[p] < min_cells: continue
            deltas_dict[p] = (pert_sums[p] / pert_counts[p]).astype(np.float32) - ctrl_mean
        del pert_sums, ctrl_sum; gc.collect()
        return deltas_dict

    def process_scperturb_h5ad(path, dataset_name):
        file_size_gb = os.path.getsize(path) / 1e9; use_backed = file_size_gb > 3.0
        print(f'\n  --- Processing {dataset_name} ({file_size_gb:.1f} GB) ---')
        if use_backed:
            print(f'  Using BACKED (memory-mapped) mode — RAM safe')
            adata = sc.read_h5ad(path, backed='r')
        else:
            adata = sc.read_h5ad(path)
        print(f'  Shape: {adata.shape}')
        X_sample = adata.X[:100]
        if sp.issparse(X_sample): X_sample = X_sample.toarray()
        is_raw = X_sample.max() > 50; del X_sample
        sc_genes = list(adata.var_names)
        pert_col = None
        for c in ['perturbation', 'gene', 'pert_symbol', 'target_gene', 'perturbation_name']:
            if c in adata.obs.columns: pert_col = c; break
        if pert_col is None:
            for c in adata.obs.columns:
                col = adata.obs[c]
                if (col.dtype == object or str(col.dtype) == 'category') and 50 < col.nunique() < 20000:
                    pert_col = c; break
        assert pert_col is not None
        pert_labels = adata.obs[pert_col].astype(str)
        ctrl_mask = pert_labels.str.lower().str.contains('control|non.targeting', regex=True, na=False)
        if 'is_control' in adata.obs.columns:
            ctrl_mask = ctrl_mask | adata.obs['is_control'].astype(bool)
        ctrl_idx = np.where(ctrl_mask.values)[0]
        if len(ctrl_idx) == 0:
            if use_backed: adata.file.close()
            return {}, sc_genes
        if use_backed:
            deltas_dict = _stream_perturbation_deltas(adata, pert_labels, ctrl_mask, is_raw, cfg.REPLOGLE_MIN_CELLS)
            adata.file.close()
        else:
            ctrl_mean = _chunked_mean(adata.X, ctrl_idx, is_raw); gc.collect()
            unique_perts = [p for p in pert_labels.unique() if not ctrl_mask[pert_labels == p].all()]
            deltas_dict = {}
            for p in tqdm(unique_perts, desc=f'  {dataset_name} deltas'):
                p_mask = (pert_labels == p) & (~ctrl_mask); p_idx = np.where(p_mask.values)[0]
                if len(p_idx) < cfg.REPLOGLE_MIN_CELLS: continue
                deltas_dict[p] = (_chunked_mean(adata.X, p_idx, is_raw) - ctrl_mean).astype(np.float32)
            del adata
        gc.collect()
        print(f'  {dataset_name}: {len(deltas_dict)} perturbations')
        return deltas_dict, sc_genes

    all_replogle_deltas = {}; all_sc_genes = None
    for name, url in SCPERTURB_URLS.items():
        if 'essential' in name.lower() and not cfg.REPLOGLE_USE_ESSENTIAL: continue
        if 'gwps' in name.lower() and not cfg.REPLOGLE_USE_GWPS: continue
        h5_path = download_h5ad(name, url)
        deltas_dict, sc_genes = process_scperturb_h5ad(h5_path, name)
        if all_sc_genes is None: all_sc_genes = sc_genes
        for p, d in deltas_dict.items():
            source_tag = 'ess' if 'essential' in name.lower() else 'gwps'
            all_replogle_deltas[f'{source_tag}_{p}'] = (d, source_tag)
        del deltas_dict; gc.collect()
        if os.path.exists(h5_path): os.remove(h5_path)

    # Map to competition gene space
    sc_gene_upper = {g.upper(): i for i, g in enumerate(all_sc_genes)}
    comp_to_replogle = {}
    for ci, g in enumerate(gene_names):
        gu = g.upper()
        if gu in sc_gene_upper: comp_to_replogle[ci] = sc_gene_upper[gu]

    replogle_deltas_list, replogle_gene_indices_list, replogle_names_list = [], [], []
    for rpert_name, (rpert_delta, source_tag) in all_replogle_deltas.items():
        actual_gene = rpert_name.split('_', 1)[1] if '_' in rpert_name else rpert_name
        comp_gi = -1
        if actual_gene in gene_to_idx: comp_gi = gene_to_idx[actual_gene]
        elif actual_gene.upper() in gene_names_upper: comp_gi = gene_to_idx[gene_names_upper[actual_gene.upper()]]
        if comp_gi < 0: continue
        mapped_delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        for comp_ci, rep_ci in comp_to_replogle.items():
            if rep_ci < len(rpert_delta): mapped_delta[comp_ci] = rpert_delta[rep_ci]
        replogle_deltas_list.append(mapped_delta)
        replogle_gene_indices_list.append(comp_gi)
        replogle_names_list.append(rpert_name)

    del all_replogle_deltas; gc.collect()
    replogle_deltas = np.array(replogle_deltas_list, dtype=np.float32)
    replogle_gene_indices = np.array(replogle_gene_indices_list, dtype=np.int64)
    replogle_names = replogle_names_list
    n_replogle = len(replogle_deltas)

    np.savez_compressed(REPLOGLE_CACHE_VFINAL, deltas=replogle_deltas,
                        gene_indices=replogle_gene_indices,
                        names=np.array(replogle_names, dtype=object))
    print(f'\n  ✓ Cached {n_replogle} Replogle perturbations to Drive')

n_replogle = len(replogle_deltas)

# Build Replogle weights and sample weights
avg_weights = weights_array.mean(axis=0)
replogle_weights = np.tile(avg_weights, (n_replogle, 1))
replogle_sample_weights = np.full(n_replogle, cfg.REPLOGLE_SAMPLE_WEIGHT, dtype=np.float32)
for i, name in enumerate(replogle_names):
    if name.startswith('gwps_'): replogle_sample_weights[i] *= cfg.REPLOGLE_GWPS_DISCOUNT

# === Diagnostics ===
n_ess = sum(1 for n in replogle_names if n.startswith('ess_'))
n_gwps = sum(1 for n in replogle_names if n.startswith('gwps_'))

# Check coverage of final test genes
test_gene_set = set(all_gene_indices[60:])
replogle_gene_set = set(replogle_gene_indices.tolist())
test_covered_by_replogle = len(test_gene_set & replogle_gene_set)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Replogle K562 Augmentation (Run B)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(['Essential', 'GWPS', 'Total'], [n_ess, n_gwps, n_replogle],
       color=['#4CAF50', '#FF9800', '#2196F3'], alpha=0.85, edgecolor='white')
for i, v in enumerate([n_ess, n_gwps, n_replogle]):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
ax.set_title(f'Replogle Source Breakdown')

ax = axes[1]
repl_abs = np.abs(replogle_deltas).mean(axis=1)
gt_abs = np.abs(deltas).mean(axis=1)
ax.hist(repl_abs, bins=60, alpha=0.6, color='#4CAF50', label=f'Replogle (n={n_replogle})', density=True)
ax.hist(gt_abs, bins=30, alpha=0.6, color='#E91E63', label=f'GT (n={len(pert_names)})', density=True)
ax.set_title('Delta Magnitude: Replogle vs GT')
ax.set_xlabel('Mean |Delta|'); ax.legend(fontsize=8)

ax = axes[2]
ax.axis('off')
repl_text = (
    f'Replogle Summary\n{"─"*40}\n'
    f'Essential: {n_ess:,}\n'
    f'GWPS:      {n_gwps:,}\n'
    f'Total:     {n_replogle:,}\n'
    f'{"─"*40}\n'
    f'Sample weight: {cfg.REPLOGLE_SAMPLE_WEIGHT}\n'
    f'GWPS discount: x{cfg.REPLOGLE_GWPS_DISCOUNT}\n'
    f'{"─"*40}\n'
    f'Final test gene coverage:\n'
    f'  By Replogle: {test_covered_by_replogle}/60\n'
    f'  (genes with real KO data)\n'
)
ax.text(0.05, 0.95, repl_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))

plt.tight_layout(); plt.show()
print(f'Replogle ready: {n_replogle:,} perturbations ({n_ess} ess + {n_gwps} gwps)')
print(f'Final test gene coverage by Replogle: {test_covered_by_replogle}/60')

In [ ]:
# ============================================================
# CELL 10: Model + Loss + Training Function (V20 — FROZEN)
#
# LightMLP: V17 architecture (Embedding → 2-layer MLP → 5127)
# cosine_light loss: WMAE - λ*cos (λ=0.08, cos_right=0.0405)
# train_v20(): Full-batch training with GT + PB + LOO + optional Replogle
# get_fold_pb/loo/replogle(): fold-safe data getters
# ============================================================
print('=' * 70)
print('CELL 10: V20 Model + Loss + Training (FROZEN)')
print('=' * 70)

# ==================== LOSS FUNCTION ====================

def cosine_light_loss(pred, target, weights, sample_weights=None,
                      lam=None, cos_right=None):
    if lam is None: lam = cfg.COSINE_LAMBDA
    if cos_right is None: cos_right = cfg.COS_RIGHT
    per_sample = (weights * torch.abs(pred - target)).mean(dim=1)
    if sample_weights is not None:
        per_sample = per_sample * sample_weights
    w_mae = per_sample.mean()
    x = torch.maximum(pred.abs(), target.abs())
    t = (x / cos_right).clamp(0, 1)
    sw = t * t * (3 - 2 * t)
    cos = F.cosine_similarity(pred * sw, target * sw, dim=-1).mean()
    return w_mae - lam * cos

# ==================== MODEL (V17 LightMLP — FROZEN) ====================

class LightMLP(nn.Module):
    def __init__(self, n_genes, hidden=256, dropout=0.3, depth=2, embed_dim=64):
        super().__init__()
        self.n_genes = n_genes
        self.gene_embed = nn.Embedding(n_genes + 1, embed_dim)
        layers = []
        in_dim = embed_dim
        for _ in range(depth):
            layers.extend([nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden),
                          nn.ReLU(), nn.Dropout(dropout)])
            in_dim = hidden
        layers.append(nn.Linear(in_dim, n_genes))
        self.mlp = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.gene_embed.weight, std=0.02)

    def forward(self, gene_idx, mixup_embed=None):
        if mixup_embed is not None:
            return self.mlp(mixup_embed)
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.mlp(self.gene_embed(safe_idx))

    def get_embed(self, gene_idx):
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.gene_embed(safe_idx)

# ==================== TRAINING FUNCTION ====================

def train_v20(model, pb_indices, pb_deltas_arr, pb_weights_arr, pb_sample_w,
              gt_indices, gt_deltas_arr, gt_weights_arr, gt_upweight,
              loo_indices, loo_deltas_arr, loo_weights_arr, loo_sample_w,
              val_indices, val_deltas, val_weights, val_bl_wmae,
              repl_indices=None, repl_deltas_arr=None, repl_weights_arr=None, repl_sample_w=None,
              weight_decay=None, epochs=None, lr=None, patience=None,
              device='cuda'):
    """V20 training: GT + PB + LOO + optional Replogle. Full-batch, no DataLoader."""
    epochs = epochs or cfg.MLP_EPOCHS
    lr = lr or cfg.MLP_LR
    patience = patience or cfg.EARLY_STOP
    wd = weight_decay if weight_decay is not None else cfg.WEIGHT_DECAY

    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 100)

    best_score, best_state, pat_count = -float('inf'), None, 0
    amp_on = USE_AMP and str(device).startswith('cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

    # Combine PB + GT + LOO + Replogle data
    parts_i = [pb_indices, gt_indices, loo_indices]
    parts_d = [pb_deltas_arr, gt_deltas_arr, loo_deltas_arr]
    parts_w = [pb_weights_arr, gt_weights_arr, loo_weights_arr]
    parts_sw = [
        pb_sample_w.astype(np.float32) if len(pb_sample_w) > 0 else np.array([], dtype=np.float32),
        np.full(len(gt_indices), gt_upweight, dtype=np.float32),
        loo_sample_w.astype(np.float32) if len(loo_sample_w) > 0 else np.array([], dtype=np.float32),
    ]

    if repl_indices is not None and len(repl_indices) > 0:
        parts_i.append(repl_indices)
        parts_d.append(repl_deltas_arr)
        parts_w.append(repl_weights_arr)
        parts_sw.append(repl_sample_w.astype(np.float32))

    all_i = np.concatenate(parts_i)
    all_d = np.concatenate(parts_d, axis=0).astype(np.float32)
    all_w = np.concatenate(parts_w, axis=0).astype(np.float32)
    all_sw = np.concatenate(parts_sw)

    va_i = torch.tensor(val_indices, dtype=torch.long, device=device)

    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(all_i))
        tr_i = torch.tensor(all_i[perm], dtype=torch.long, device=device)
        tr_d = torch.tensor(all_d[perm], dtype=torch.float32, device=device)
        tr_w = torch.tensor(all_w[perm], dtype=torch.float32, device=device)
        tr_sw = torch.tensor(all_sw[perm], dtype=torch.float32, device=device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_on):
            pred = model(tr_i)
            loss = cosine_light_loss(pred, tr_d, tr_w, tr_sw)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()

        del tr_i, tr_d, tr_w, tr_sw

        # Validation every 10 epochs
        if (ep + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=amp_on):
                    vp = model(va_i).float().cpu().numpy()
            s, _, _ = metric.score(vp, val_deltas, val_weights, val_bl_wmae)
            if s > best_score:
                best_score = s
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                pat_count = 0
            else:
                pat_count += 10
            if pat_count >= patience:
                break

    if best_state: model.load_state_dict(best_state)

    del va_i; clean_memory()
    return model.cpu(), best_score


def get_fold_pb(val_pert_set):
    """Return PB samples excluding val fold perturbations."""
    mask = np.array([pn not in val_pert_set for pn in pb_pert_names])
    if mask.any():
        assert len(set(np.array(pb_pert_names)[mask]) & val_pert_set) == 0, 'LEAKAGE!'
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (pb_pert_indices[mask], pb_deltas[mask], pb_weights[mask],
            pb_cell_weights[mask])


def get_fold_loo(loo_gene_indices_arr, loo_deltas_arr, loo_sw_arr, val_pert_gene_set):
    """Return LOO samples excluding val fold gene indices (prevent leakage)."""
    mask = np.array([gi not in val_pert_gene_set for gi in loo_gene_indices_arr])
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    avg_w = weights_array.mean(axis=0)
    loo_w = np.tile(avg_w, (mask.sum(), 1))
    return (loo_gene_indices_arr[mask], loo_deltas_arr[mask], loo_w, loo_sw_arr[mask])


def get_fold_replogle(val_pert_gene_set):
    """Return Replogle samples excluding val fold gene indices."""
    mask = np.array([gi not in val_pert_gene_set for gi in replogle_gene_indices])
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (replogle_gene_indices[mask], replogle_deltas[mask],
            replogle_weights[mask], replogle_sample_weights[mask])


# --- Model summary ---
test_model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                      dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                      embed_dim=cfg.EMBED_DIM)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'\nLightMLP params: {n_params:,}')
del test_model

print(f'Loss: cosine_light (lambda={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT})')
print(f'Training: train_v20() with GT + PB + LOO + optional Replogle')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 11: Final Training — Run A & B for ALL 120 Perturbations
#
# KEY CHANGES FROM V23:
# - Predicts ALL 120 perturbations (not just 60)
# - KO correction applied to ALL 120 using pert_ids_all.csv
# - NO zeros for rows 61-120 (real predictions!)
# - Each run checkpointed to Drive for resume
# ============================================================
print('=' * 70)
print('CELL 11: Final Training — ALL 120 Perturbations')
print('=' * 70)

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
all_submissions = {}  # run_id -> DataFrame
all_cv_scores = {}    # run_id -> cv_score

for run_id, run_cfg in RUN_CONFIGS.items():
    run_name = run_cfg['name']
    use_replogle = run_cfg['replogle']

    sub_cache = os.path.join(VFINAL_DIR, f'submission_run_{run_id}.csv')

    # === Resume gate ===
    if os.path.exists(sub_cache):
        print(f'\n✓ RESUME: Run {run_id} ({run_name}) — loading cached submission')
        sub_df = pd.read_csv(sub_cache)
        all_submissions[run_id] = sub_df
        cv_cache = load_json(f'cv_run_{run_id}')
        if cv_cache:
            all_cv_scores[run_id] = cv_cache
            print(f'  CV={cv_cache.get("cv_mean", "?"):.4f}')
        continue

    print(f'\n{"="*60}')
    print(f'  Run {run_id}: {run_name}')
    print(f'  Replogle: {use_replogle}')
    print(f'{"="*60}')

    # === 5-fold CV first (to get CV score) ===
    cv_cache = load_json(f'cv_run_{run_id}')
    if cv_cache:
        print(f'  ✓ CV already computed: {cv_cache.get("cv_mean", "?"):.4f}')
        all_cv_scores[run_id] = cv_cache
    else:
        print(f'  Running 5-fold CV...')
        kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
        fold_splits = list(kf.split(deltas))
        oof_preds = np.zeros_like(deltas, dtype=np.float32)
        fold_scores = []
        t0_cv = time.time()

        for fold, (train_idx, val_idx) in enumerate(fold_splits):
            # Check fold checkpoint
            fold_ckpt_path = os.path.join(VFINAL_DIR, f'run_{run_id}_fold{fold}_preds.npz')
            if os.path.exists(fold_ckpt_path):
                fd = np.load(fold_ckpt_path)
                oof_preds[val_idx] = fd['preds']
                fold_scores.append(float(fd['score']))
                print(f'    ✓ Fold {fold+1}: resumed (score={float(fd["score"]):.4f})')
                continue

            fold_t0 = time.time()
            val_pert_set = {pert_names[i] for i in val_idx}
            val_gene_set = {pert_gene_indices[i] for i in val_idx if pert_gene_indices[i] >= 0}

            gt_gi = pert_gene_indices[train_idx]; gt_y = deltas[train_idx]; gt_w = weights_array[train_idx]
            pb_gi_f, pb_y_f, pb_w_f, pb_sw_f = get_fold_pb(val_pert_set)
            loo_gi_f, loo_y_f, loo_w_f, loo_sw_f = get_fold_loo(
                loo_gene_indices, loo_deltas_all, loo_sample_weights_all, val_gene_set)

            repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = None, None, None, None
            if use_replogle:
                repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = get_fold_replogle(val_gene_set)

            val_gi = pert_gene_indices[val_idx]; val_y = deltas[val_idx]
            val_w = weights_array[val_idx]; val_bl = baseline_wmae[val_idx]

            n_repl = len(repl_gi_f) if repl_gi_f is not None else 0
            print(f'    Fold {fold+1}: {len(gt_gi)} GT + {len(pb_gi_f)} PB + {len(loo_gi_f)} LOO + {n_repl} Repl')

            fold_ensemble = np.zeros((len(val_idx), cfg.NUM_GENES), dtype=np.float32)
            for ens in range(cfg.N_MLP_ENSEMBLE):
                seed = cfg.SEED + ord(run_id) * 10000 + fold * 1000 + ens * 7
                torch.manual_seed(seed); np.random.seed(seed)
                model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                                 dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH, embed_dim=cfg.EMBED_DIM)
                model, _ = train_v20(model, pb_gi_f, pb_y_f, pb_w_f, pb_sw_f,
                    gt_gi, gt_y, gt_w, cfg.GT_UPWEIGHT,
                    loo_gi_f, loo_y_f, loo_w_f, loo_sw_f,
                    val_gi, val_y, val_w, val_bl,
                    repl_indices=repl_gi_f, repl_deltas_arr=repl_y_f,
                    repl_weights_arr=repl_w_f, repl_sample_w=repl_sw_f,
                    device=cfg.DEVICE)
                model.eval()
                with torch.no_grad():
                    vp = model(torch.tensor(val_gi, dtype=torch.long)).numpy()
                fold_ensemble += vp / cfg.N_MLP_ENSEMBLE
                del model; clean_memory()

            # KO correction
            for i, vi in enumerate(val_idx):
                gi = pert_gene_indices[vi]
                if gi >= 0:
                    fold_ensemble[i] = apply_ko(fold_ensemble[i], gi, knockout_stats, cfg.KNOCKOUT_MULTIPLIER)

            oof_preds[val_idx] = fold_ensemble
            fold_score, fold_w, fold_wcos = metric.score(fold_ensemble, val_y, val_w, val_bl)
            fold_scores.append(fold_score)
            elapsed = time.time() - fold_t0
            print(f'    Fold {fold+1}: {fold_score:.4f} (W={fold_w:.2f}, wcos={fold_wcos:.4f}) [{elapsed/60:.1f} min]')

            # Save fold checkpoint
            np.savez_compressed(fold_ckpt_path, preds=fold_ensemble, score=fold_score)

        cv_score, cv_w, cv_wcos = metric.score(oof_preds, deltas, weights_array, baseline_wmae)
        cv_result = {'cv_mean': float(np.mean(fold_scores)), 'cv_std': float(np.std(fold_scores)),
                     'cv_score': float(cv_score), 'cv_w': float(cv_w), 'cv_wcos': float(cv_wcos),
                     'fold_scores': [float(s) for s in fold_scores],
                     'cv_time_min': (time.time() - t0_cv) / 60}
        save_json(f'cv_run_{run_id}', cv_result)
        all_cv_scores[run_id] = cv_result
        print(f'\n  Run {run_id} CV = {cv_result["cv_mean"]:.4f} ± {cv_result["cv_std"]:.4f}')

    # === Final Training: predict ALL 120 perturbations ===
    print(f'\n  --- Final Training: ALL 120 perts ---')

    # 90/10 early stopping split
    n_es = max(8, len(pert_names) // 10)
    es_idx = np.random.RandomState(cfg.SEED).choice(len(pert_names), size=n_es, replace=False)
    tr_idx_final = np.array([i for i in range(len(pert_names)) if i not in es_idx])

    es_pert_set = {pert_names[i] for i in es_idx}
    es_gene_set = {pert_gene_indices[i] for i in es_idx if pert_gene_indices[i] >= 0}

    pb_i_f, pb_d_f, pb_w_f, pb_sw_f = get_fold_pb(es_pert_set)
    loo_gi_f, loo_y_f, loo_w_f, loo_sw_f = get_fold_loo(
        loo_gene_indices, loo_deltas_all, loo_sample_weights_all, es_gene_set)

    repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = None, None, None, None
    if use_replogle:
        repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = get_fold_replogle(es_gene_set)

    tr_gi = pert_gene_indices[tr_idx_final]; tr_y = deltas[tr_idx_final]; tr_w = weights_array[tr_idx_final]
    es_gi = pert_gene_indices[es_idx]; es_y = deltas[es_idx]
    es_w = weights_array[es_idx]; es_bl = baseline_wmae[es_idx]

    n_repl = len(repl_gi_f) if repl_gi_f is not None else 0
    print(f'  Train: {len(tr_gi)} GT + {len(pb_i_f)} PB + {len(loo_gi_f)} LOO + {n_repl} Repl')
    print(f'  ES holdout: {n_es} perts')

    t0 = time.time()
    # Predict for ALL 120 perturbation gene indices
    final_preds = np.zeros((120, cfg.NUM_GENES), dtype=np.float32)

    for m_i in tqdm(range(cfg.N_MLP_ENSEMBLE), desc=f'Run {run_id} training'):
        seed = cfg.SEED + ord(run_id) * 10000 + 9999 + m_i * 7
        torch.manual_seed(seed); np.random.seed(seed)
        model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                         dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH, embed_dim=cfg.EMBED_DIM)
        model, _ = train_v20(model, pb_i_f, pb_d_f, pb_w_f, pb_sw_f,
            tr_gi, tr_y, tr_w, cfg.GT_UPWEIGHT,
            loo_gi_f, loo_y_f, loo_w_f, loo_sw_f,
            es_gi, es_y, es_w, es_bl,
            repl_indices=repl_gi_f, repl_deltas_arr=repl_y_f,
            repl_weights_arr=repl_w_f, repl_sample_w=repl_sw_f,
            device=cfg.DEVICE)
        model.eval()
        with torch.no_grad():
            # Predict for ALL 120 perturbation gene indices
            pred_all = model(torch.tensor(all_gene_indices, dtype=torch.long)).numpy()
        final_preds += pred_all
        del model; clean_memory()

    final_preds /= cfg.N_MLP_ENSEMBLE
    elapsed = (time.time() - t0) / 60

    # === KO correction for ALL 120 perturbations ===
    n_ko_applied = 0
    for pi in range(120):
        gene_idx = all_gene_indices[pi]
        final_preds[pi] = apply_ko(final_preds[pi], gene_idx, knockout_stats)
        if gene_idx >= 0: n_ko_applied += 1

    print(f'  KO correction applied to {n_ko_applied}/120 perturbations')

    # CRITICAL: NO zeroing of rows 61-120! Real predictions for all 120.
    assert final_preds[60:].any(), 'FATAL: rows 61-120 are all zeros — KO correction failed!'

    # Build submission DataFrame
    sub = pd.DataFrame(final_preds, columns=gene_names)
    sub.insert(0, 'pert_id', all_pert_ids)
    assert list(sub.columns) == SUBMISSION_COLUMNS
    assert len(sub) == 120
    assert not sub.isnull().any().any()

    # Save to Drive
    sub.to_csv(sub_cache, index=False)
    all_submissions[run_id] = sub
    print(f'  ✓ Run {run_id} done in {elapsed:.1f} min — 120 rows with real predictions')

# === Summary Chart ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('V_FINAL Training Complete — ALL 120 Perturbations', fontsize=14, fontweight='bold')

# Chart 1: CV scores comparison
ax = axes[0]
run_labels = list(RUN_CONFIGS.keys())
cv_means = [all_cv_scores.get(r, {}).get('cv_mean', 0) for r in run_labels]
cv_stds = [all_cv_scores.get(r, {}).get('cv_std', 0) for r in run_labels]
colors = ['#2196F3', '#4CAF50']
bars = ax.bar(run_labels, cv_means, yerr=cv_stds, capsize=6, color=colors, alpha=0.85, edgecolor='white')
ax.axhline(y=0.9672, color='red', linestyle='--', linewidth=1.5, label='V20 RUN=3 CV=0.967')
ax.set_title('5-Fold CV Scores')
ax.set_ylabel('CV Score')
ax.legend(fontsize=8)
for bar, cv_m in zip(bars, cv_means):
    if cv_m > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{cv_m:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Chart 2: Prediction stats comparison (public vs private)
ax = axes[1]
for rid in run_labels:
    if rid not in all_submissions: continue
    sub = all_submissions[rid]
    pub_abs = np.abs(sub.iloc[:60, 1:].values.astype(np.float32)).mean(axis=1)
    prv_abs = np.abs(sub.iloc[60:, 1:].values.astype(np.float32)).mean(axis=1)
    ax.scatter(pub_abs.mean(), prv_abs.mean(), s=200, label=f'Run {rid}', zorder=5, edgecolors='black')
ax.plot([0, 0.1], [0, 0.1], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Mean |Pred| (public, pert_1-60)')
ax.set_ylabel('Mean |Pred| (private, pert_61-120)')
ax.set_title('Public vs Private Prediction Magnitudes')
ax.legend()

# Chart 3: Summary
ax = axes[2]
ax.axis('off')
summary = f'V_FINAL Training Summary\n{"─"*45}\n'
for rid in run_labels:
    cv = all_cv_scores.get(rid, {}).get('cv_mean', 0)
    desc = RUN_CONFIGS[rid]['desc']
    summary += f'  Run {rid}: CV={cv:.4f}\n    {desc}\n'
summary += f'\n{"─"*45}\n'
summary += f'Predictions: ALL 120 perturbations\n'
summary += f'  Public (1-60):  REAL predictions\n'
summary += f'  Private (61-120): REAL predictions\n'
summary += f'  NO ZEROS anywhere!\n'
summary += f'Timestamp: {ts}\n'
ax.text(0.02, 0.95, summary, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout(); plt.show()
print(f'\n{"="*60}')
print('ALL RUNS COMPLETE — 120-row predictions generated')
print(f'{"="*60}')

In [ ]:
# ============================================================
# CELL 12: Score Prediction, Correlation Check, Ensemble (Run C)
#
# - Loads V20 RUN=3 reference for correlation check
# - Predicts scores for pert_1-60, pert_61-120, pert_1-120
# - Creates Run C = 50/50 ensemble of A + B
# - Saves all 3 submission files with proper naming
# ============================================================
print('=' * 70)
print('CELL 12: Score Prediction + Correlation + Ensemble')
print('=' * 70)

# === Load V20 RUN=3 reference from Drive ===
V20_REF_NAME = 'submission_v20_loo_20260222_180450_0_9672x3.csv'
v20_ref_path = os.path.join(OUTPUT_DIR, V20_REF_NAME)
if not os.path.exists(v20_ref_path):
    # Try other locations
    for candidate in [os.path.join(BASE_DIR, V20_REF_NAME),
                      os.path.join('/content', V20_REF_NAME)]:
        if os.path.exists(candidate):
            v20_ref_path = candidate; break

v20_ref = None
if os.path.exists(v20_ref_path):
    v20_ref = pd.read_csv(v20_ref_path)
    print(f'✓ V20 RUN=3 reference loaded: {v20_ref.shape}')
    v20_pub = v20_ref.iloc[:60, 1:].values.astype(np.float64).ravel()
    print(f'  Public rows: mean={v20_pub.mean():.6f}, std={np.std(v20_pub):.6f}')
else:
    print(f'⚠ V20 reference not found at {v20_ref_path}')
    print(f'  Upload {V20_REF_NAME} to Drive for correlation check')

# === Correlation check: Run A pert_1-60 vs V20 RUN=3 ===
print(f'\n--- Correlation with V20 RUN=3 Reference ---')
corr_results = {}
for rid in all_submissions:
    sub = all_submissions[rid]
    pub_preds = sub.iloc[:60, 1:].values.astype(np.float64).ravel()
    prv_preds = sub.iloc[60:, 1:].values.astype(np.float64).ravel()

    if v20_ref is not None:
        corr_pub = np.corrcoef(pub_preds, v20_pub)[0, 1]
    else:
        corr_pub = None

    pub_stats = {'mean': pub_preds.mean(), 'std': np.std(pub_preds),
                 'abs_mean': np.abs(pub_preds).mean(), 'zeros_pct': 100*np.mean(np.abs(pub_preds) < 1e-8)}
    prv_stats = {'mean': prv_preds.mean(), 'std': np.std(prv_preds),
                 'abs_mean': np.abs(prv_preds).mean(), 'zeros_pct': 100*np.mean(np.abs(prv_preds) < 1e-8)}

    corr_results[rid] = {'corr_v20_pub': corr_pub, 'pub': pub_stats, 'prv': prv_stats}

    corr_str = f'{corr_pub:.6f}' if corr_pub is not None else 'N/A'
    status = ''
    if corr_pub is not None:
        if corr_pub > 0.999: status = ' ✓ EXCELLENT'
        elif corr_pub > 0.995: status = ' ✓ GOOD'
        elif corr_pub > 0.99: status = ' ~ OK'
        else: status = ' ⚠ LOW — investigate!'
    print(f'  Run {rid}: corr(pub, V20)={corr_str}{status}')
    print(f'    Public:  mean={pub_stats["abs_mean"]:.5f}, std={pub_stats["std"]:.5f}, zeros={pub_stats["zeros_pct"]:.1f}%')
    print(f'    Private: mean={prv_stats["abs_mean"]:.5f}, std={prv_stats["std"]:.5f}, zeros={prv_stats["zeros_pct"]:.1f}%')

# === Create Run C: 50/50 Ensemble ===
print(f'\n--- Creating Run C: 50/50 Ensemble of A + B ---')
if 'A' in all_submissions and 'B' in all_submissions:
    sub_a = all_submissions['A'].iloc[:, 1:].values.astype(np.float64)
    sub_b = all_submissions['B'].iloc[:, 1:].values.astype(np.float64)
    ensemble_preds = 0.5 * sub_a + 0.5 * sub_b

    sub_c = pd.DataFrame(ensemble_preds.astype(np.float32), columns=gene_names)
    sub_c.insert(0, 'pert_id', all_pert_ids)
    assert list(sub_c.columns) == SUBMISSION_COLUMNS
    assert len(sub_c) == 120

    all_submissions['C'] = sub_c
    # Correlation of ensemble with V20
    if v20_ref is not None:
        ens_pub = ensemble_preds[:60].ravel()
        corr_c = np.corrcoef(ens_pub, v20_pub)[0, 1]
        prv_c = ensemble_preds[60:].ravel()
        corr_results['C'] = {
            'corr_v20_pub': corr_c,
            'pub': {'mean': ens_pub.mean(), 'std': np.std(ens_pub), 'abs_mean': np.abs(ens_pub).mean()},
            'prv': {'mean': prv_c.mean(), 'std': np.std(prv_c), 'abs_mean': np.abs(prv_c).mean()},
        }
        print(f'  Run C: corr(pub, V20)={corr_c:.6f}')
    print(f'  ✓ Ensemble created: {sub_c.shape}')
else:
    print(f'  ⚠ Cannot create ensemble — need both Run A and Run B')

# === Score Prediction (estimated from CV + historical patterns) ===
print(f'\n--- Score Prediction ---')
# Historical CV → LB data points
historical_cv_lb = [
    (0.967, 4.090), (0.969, 4.068), (0.964, 4.066),  # V20 family
    (0.943, 4.064), (0.948, 4.051), (0.981, 4.033),   # V23 R5, V22, V20 R4
    (0.959, 3.985), (0.968, 3.874), (0.977, 3.855),   # V23 R3, R6, R1
    (0.931, 3.792), (0.977, 3.772), (0.860, 3.590),   # V21, V23 R4, R2
]
hist_cv = np.array([x[0] for x in historical_cv_lb])
hist_lb = np.array([x[1] for x in historical_cv_lb])

# Note: These LB scores were on pert_1-60 only (public).
# The final score will be on pert_61-120 only (private).
# Since our model is BLIND (same prediction quality for any gene),
# we expect similar performance on pert_61-120.

# Simple prediction: use correlation with V20 RUN=3 as primary predictor
print(f'\nPublic LB Prediction (pert_1-60):')
for rid in sorted(all_submissions.keys()):
    cv = all_cv_scores.get(rid, {}).get('cv_mean', 0) if rid in all_cv_scores else 0
    corr = corr_results.get(rid, {}).get('corr_v20_pub')
    if corr is not None and corr > 0.99:
        # High correlation with V20 → expect similar LB
        pred_lb = 4.090 * corr  # Simple scaling
    elif cv > 0:
        # Fallback: rough CV→LB from historical
        pred_lb = 3.5 + 0.6 * cv  # Very rough
    else:
        pred_lb = 0
    print(f'  Run {rid}: Predicted public LB ≈ {pred_lb:.3f} (CV={cv:.4f}, corr_V20={corr:.4f} if available)')

print(f'\nPrivate Score Prediction (pert_61-120):')
print(f'  Our model is BLIND — it predicts the same delta for any gene.')
print(f'  The only per-gene element is KO correction.')
print(f'  Since KO correction is applied to all 120 perts, we expect')
print(f'  pert_61-120 quality ≈ pert_1-60 quality.')
print(f'  → Expected private score ≈ public LB (within noise)')

# === Comprehensive Charts ===
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('V_FINAL: Score Prediction & Correlation Analysis', fontsize=15, fontweight='bold')

# Chart 1: Correlation with V20 reference
ax = axes[0, 0]
rids = sorted(corr_results.keys())
corrs = [corr_results[r].get('corr_v20_pub', 0) or 0 for r in rids]
colors = ['#2196F3' if r == 'A' else '#4CAF50' if r == 'B' else '#FF9800' for r in rids]
bars = ax.bar(rids, corrs, color=colors, alpha=0.85, edgecolor='white')
ax.axhline(y=0.999, color='red', linestyle='--', alpha=0.5, label='Target: 0.999')
ax.set_title('Correlation with V20 RUN=3 (public rows)')
ax.set_ylabel('Pearson Correlation')
ax.legend(); ax.set_ylim(min(corrs) - 0.005, 1.001)
for bar, c in zip(bars, corrs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{c:.5f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Chart 2: Public vs Private prediction magnitude per perturbation
ax = axes[0, 1]
for rid in sorted(all_submissions.keys()):
    sub = all_submissions[rid]
    pub_mags = np.abs(sub.iloc[:60, 1:].values.astype(np.float32)).mean(axis=1)
    prv_mags = np.abs(sub.iloc[60:, 1:].values.astype(np.float32)).mean(axis=1)
    ax.plot(range(60), pub_mags, alpha=0.6, label=f'Run {rid} public')
    ax.plot(range(60, 120), prv_mags, alpha=0.6, linestyle='--')
ax.axvline(x=59.5, color='red', linestyle='-', linewidth=2, alpha=0.3, label='Public|Private boundary')
ax.set_title('Mean |Prediction| per Perturbation')
ax.set_xlabel('Perturbation Index')
ax.set_ylabel('Mean |Delta|')
ax.legend(fontsize=7)

# Chart 3: Distribution of public vs private predictions
ax = axes[0, 2]
for rid in sorted(all_submissions.keys()):
    sub = all_submissions[rid]
    pub_flat = sub.iloc[:60, 1:].values.astype(np.float32).ravel()
    prv_flat = sub.iloc[60:, 1:].values.astype(np.float32).ravel()
    ax.hist(pub_flat, bins=100, alpha=0.3, density=True, label=f'Run {rid} pub')
    ax.hist(prv_flat, bins=100, alpha=0.3, density=True, linestyle='--', label=f'Run {rid} prv')
ax.set_title('Prediction Distribution: Public vs Private')
ax.set_xlabel('Predicted Delta')
ax.legend(fontsize=6)
ax.set_xlim(-0.5, 0.5)

# Chart 4: Pairwise correlation heatmap (A, B, C)
ax = axes[1, 0]
sub_ids = sorted(all_submissions.keys())
n_subs = len(sub_ids)
corr_mat = np.ones((n_subs, n_subs))
for i, ri in enumerate(sub_ids):
    for j, rj in enumerate(sub_ids):
        if i >= j: continue
        # Full 120-row correlation
        pi = all_submissions[ri].iloc[:, 1:].values.ravel()
        pj = all_submissions[rj].iloc[:, 1:].values.ravel()
        corr_mat[i, j] = corr_mat[j, i] = np.corrcoef(pi, pj)[0, 1]
im = ax.imshow(corr_mat, cmap='RdYlGn', vmin=0.98, vmax=1.0)
ax.set_xticks(range(n_subs)); ax.set_xticklabels([f'Run {r}' for r in sub_ids], fontsize=9)
ax.set_yticks(range(n_subs)); ax.set_yticklabels([f'Run {r}' for r in sub_ids], fontsize=9)
for i in range(n_subs):
    for j in range(n_subs):
        ax.text(j, i, f'{corr_mat[i,j]:.4f}', ha='center', va='center', fontsize=9)
ax.set_title('Pairwise Correlation (all 120 rows)')
plt.colorbar(im, ax=ax, shrink=0.6)

# Chart 5: KO correction verification
ax = axes[1, 1]
for rid in sorted(all_submissions.keys()):
    sub = all_submissions[rid]
    ko_values = []
    for pi in range(120):
        gi = all_gene_indices[pi]
        if gi >= 0 and gi < len(gene_names):
            gene_name = gene_names[gi]
            if gene_name in sub.columns:
                ko_values.append(sub.iloc[pi][gene_name])
    if ko_values:
        ax.hist(ko_values, bins=30, alpha=0.5, label=f'Run {rid} (n={len(ko_values)})')
ax.axvline(knockout_stats['mean'], color='red', linestyle='--', linewidth=2,
           label=f'Expected KO={knockout_stats["mean"]:.3f}')
ax.set_title('KO Correction Verification (self-targeting gene)')
ax.set_xlabel('Predicted Self-Targeting Delta')
ax.legend(fontsize=8)

# Chart 6: Summary text
ax = axes[1, 2]
ax.axis('off')
txt = f'V_FINAL Score Prediction\n{"─"*50}\n\n'
txt += f'PUBLIC LB (pert_1-60):\n'
for rid in sorted(all_submissions.keys()):
    cv = all_cv_scores.get(rid, {}).get('cv_mean', 0) if rid in all_cv_scores else 0
    corr = corr_results.get(rid, {}).get('corr_v20_pub')
    corr_s = f'{corr:.5f}' if corr else 'N/A'
    txt += f'  Run {rid}: CV={cv:.4f}, corr_V20={corr_s}\n'
txt += f'\n{"─"*50}\n'
txt += f'PRIVATE (pert_61-120) — FINAL RANKING:\n'
txt += f'  Model is blind → same quality as public\n'
txt += f'  KO correction applied to all 120 perts\n'
txt += f'  Expected: ≈ public LB score\n'
txt += f'\n{"─"*50}\n'
txt += f'RECOMMENDATION:\n'
txt += f'  Submit Run A (safest) + Run C (hedged)\n'
txt += f'  or all 3 if allowed\n'
ax.text(0.02, 0.98, txt, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout(); plt.show()
print('✓ Analysis complete')

In [ ]:
# ============================================================
# CELL 13: Save Submission Files + Final Report + Zip + Download
#
# - Saves properly named submission CSVs for Kaggle upload
# - Generates comprehensive HTML report
# - Zips everything (submissions, report, configs, charts)
# - Downloads the zip file
# ============================================================
print('=' * 70)
print('CELL 13: Final Submission Files + Report + Zip + Download')
print('=' * 70)

ts_final = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
PACKAGE_DIR = f'/content/vfinal_package_{ts_final}'
os.makedirs(PACKAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(PACKAGE_DIR, 'submissions'), exist_ok=True)
os.makedirs(os.path.join(PACKAGE_DIR, 'charts'), exist_ok=True)

# === Save submission CSVs ===
saved_files = {}
for rid in sorted(all_submissions.keys()):
    sub = all_submissions[rid]
    run_cfg = RUN_CONFIGS.get(rid, {'name': f'Ensemble_{rid}'})
    cv = all_cv_scores.get(rid, {}).get('cv_mean', 0) if rid in all_cv_scores else 0
    cv_str = f'{cv:.4f}'.replace('.', '_') if cv > 0 else '0000'
    fname = f'submission_vfinal_run{rid}_{run_cfg.get("name","ens")}_{ts_final}_{cv_str}.csv'

    # Save to package
    pkg_path = os.path.join(PACKAGE_DIR, 'submissions', fname)
    sub.to_csv(pkg_path, index=False)

    # Save to Drive
    drive_path = os.path.join(OUTPUT_DIR, fname)
    sub.to_csv(drive_path, index=False)

    # Save locally for quick download
    local_path = f'/content/{fname}'
    sub.to_csv(local_path, index=False)

    saved_files[rid] = {'fname': fname, 'local': local_path, 'drive': drive_path, 'pkg': pkg_path}
    print(f'  ✓ Run {rid}: {fname}')

# === Verify all submissions ===
print(f'\n--- Submission Verification ---')
all_ok = True
for rid, info in saved_files.items():
    sub = pd.read_csv(info['local'])
    checks = {
        'shape': sub.shape == (120, len(SUBMISSION_COLUMNS)),
        'columns': list(sub.columns) == SUBMISSION_COLUMNS,
        'no_nulls': not sub.isnull().any().any(),
        'pub_nonzero': sub.iloc[:60, 1:].values.any(),
        'prv_nonzero': sub.iloc[60:, 1:].values.any(),
    }
    status = '✓' if all(checks.values()) else '✗'
    print(f'  Run {rid}: {status} shape={sub.shape}, all checks={all(checks.values())}')
    if not all(checks.values()):
        for k, v in checks.items():
            if not v: print(f'    FAIL: {k}')
        all_ok = False

if all_ok:
    print('  ✓ ALL submissions verified — 120 rows, no zeros in private rows')
else:
    print('  ⚠ Some checks failed — review above')

# === Generate HTML Report ===
print(f'\n--- Generating HTML Report ---')

# Collect all stats for report
report_data = {}
for rid in sorted(all_submissions.keys()):
    sub = all_submissions[rid]
    pub_vals = sub.iloc[:60, 1:].values.astype(np.float64)
    prv_vals = sub.iloc[60:, 1:].values.astype(np.float64)
    all_vals = sub.iloc[:, 1:].values.astype(np.float64)

    report_data[rid] = {
        'name': RUN_CONFIGS.get(rid, {'name': f'Ensemble_{rid}'}).get('name', rid),
        'desc': RUN_CONFIGS.get(rid, {'desc': '50/50 ensemble'}).get('desc', ''),
        'cv_mean': all_cv_scores.get(rid, {}).get('cv_mean', 0) if rid in all_cv_scores else 0,
        'cv_std': all_cv_scores.get(rid, {}).get('cv_std', 0) if rid in all_cv_scores else 0,
        'corr_v20': corr_results.get(rid, {}).get('corr_v20_pub', None),
        'pub_mean_abs': np.abs(pub_vals).mean(),
        'pub_std': pub_vals.std(),
        'prv_mean_abs': np.abs(prv_vals).mean(),
        'prv_std': prv_vals.std(),
        'all_mean_abs': np.abs(all_vals).mean(),
        'fname': saved_files[rid]['fname'],
    }

html = f"""<!DOCTYPE html>
<html><head><title>V_FINAL Submission Report</title>
<style>
body {{ font-family: 'Segoe UI', Arial, sans-serif; max-width: 1200px; margin: 0 auto; padding: 20px; background: #f5f5f5; }}
h1 {{ color: #1a237e; border-bottom: 3px solid #1a237e; padding-bottom: 10px; }}
h2 {{ color: #283593; margin-top: 30px; }}
.card {{ background: white; border-radius: 8px; padding: 20px; margin: 15px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }}
.critical {{ background: #fff3e0; border-left: 4px solid #ff9800; }}
.success {{ background: #e8f5e9; border-left: 4px solid #4caf50; }}
.info {{ background: #e3f2fd; border-left: 4px solid #2196f3; }}
table {{ border-collapse: collapse; width: 100%; margin: 10px 0; }}
th, td {{ border: 1px solid #ddd; padding: 8px 12px; text-align: center; }}
th {{ background: #1a237e; color: white; }}
tr:nth-child(even) {{ background: #f8f9fa; }}
.highlight {{ font-weight: bold; color: #1a237e; }}
code {{ background: #f0f0f0; padding: 2px 6px; border-radius: 3px; font-size: 0.9em; }}
</style></head><body>
<h1>V_FINAL: Last Submission Report</h1>
<p><strong>Generated:</strong> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
<p><strong>Base:</strong> V20 RUN=3 (LB=4.08996, ALL-TIME BEST)</p>

<div class="card critical">
<h2>Critical Context</h2>
<p><strong>ALL 120 perturbation identities are now known.</strong> Previous submissions had zeros for rows 61-120.</p>
<p>The final ranking uses <strong>ONLY pert_61-120</strong> (private test set).</p>
<p>These submissions contain <strong>real predictions for all 120 rows</strong> — no zeros.</p>
</div>

<h2>Submission Summary</h2>
<table>
<tr><th>Run</th><th>Name</th><th>CV Score</th><th>Corr with V20</th><th>Pub Mean|Δ|</th><th>Prv Mean|Δ|</th><th>File</th></tr>
"""

for rid in sorted(report_data.keys()):
    rd = report_data[rid]
    corr_s = f'{rd["corr_v20"]:.5f}' if rd['corr_v20'] else 'N/A'
    html += f'<tr><td class="highlight">Run {rid}</td><td>{rd["name"]}</td>'
    html += f'<td>{rd["cv_mean"]:.4f} ± {rd["cv_std"]:.4f}</td>'
    html += f'<td>{corr_s}</td>'
    html += f'<td>{rd["pub_mean_abs"]:.5f}</td><td>{rd["prv_mean_abs"]:.5f}</td>'
    html += f'<td><code>{rd["fname"]}</code></td></tr>\n'

html += """</table>

<div class="card success">
<h2>Score Prediction</h2>
<p><strong>Public LB (pert_1-60):</strong> Expected ≈ 4.09 (same as V20 RUN=3, high correlation)</p>
<p><strong>Private (pert_61-120):</strong> Expected ≈ public LB score</p>
<p><em>Rationale:</em> The model is <strong>blind</strong> — it predicts the same delta pattern for any gene.
The only per-gene element is KO correction, which is now applied to all 120 perturbations using the
revealed gene identities from <code>pert_ids_all.csv</code>.</p>
</div>

<h2>Configuration (FROZEN — V20 RUN=3)</h2>
<div class="card info">
<table>
<tr><th>Parameter</th><th>Value</th><th>Status</th></tr>
"""

config_items = [
    ('Architecture', 'V17 LightMLP (H=384, D=2)', 'FROZEN'),
    ('Loss', f'cosine_light (λ={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT})', 'FROZEN'),
    ('LOO Method', f'Baseline {cfg.LOO_PERCENTILE}th percentile', 'FROZEN'),
    ('LOO Resamples', str(cfg.LOO_N_RESAMPLES), 'FROZEN'),
    ('LOO Weight', str(cfg.LOO_SAMPLE_WEIGHT), 'FROZEN'),
    ('Ensemble', f'{cfg.N_MLP_ENSEMBLE} models', 'FROZEN'),
    ('KO Correction', f'mult={cfg.KNOCKOUT_MULTIPLIER}', 'FROZEN'),
    ('Prediction Rows', '120 (ALL perturbations)', 'NEW'),
    ('Private Rows', 'Real predictions (not zeros)', 'NEW'),
]
for name, val, status in config_items:
    color = '#4caf50' if status == 'FROZEN' else '#ff9800'
    html += f'<tr><td>{name}</td><td>{val}</td><td style="color:{color};font-weight:bold">{status}</td></tr>\n'

html += """</table></div>

<h2>Final Test Perturbations (pert_61-120)</h2>
<div class="card">
<p>60 genes — <strong>32% chromatin/epigenetic enrichment:</strong></p>
<p style="font-family:monospace;font-size:0.85em">"""

test_genes_list = [all_pert_symbols[i] for i in range(60, 120) if all_pert_symbols[i]]
html += ', '.join(test_genes_list)
html += """</p>
<p><strong>Key properties:</strong></p>
<ul>
<li>0 overlap with training (80 genes) or validation (60 genes) — all 200 genes disjoint</li>
<li>Gene family overlaps: HDAC2/3↔HDAC4/8, SIRT6↔SIRT1/2, KDM1A/2B/3B/6B↔KDM4A/5C</li>
</ul>
</div>

<h2>Recommendation</h2>
<div class="card success">
<ul>
<li><strong>1st submission:</strong> Run A (pure V20 reproduction — lowest risk, proven best)</li>
<li><strong>2nd submission:</strong> Run C (50/50 ensemble — hedged bet)</li>
<li><strong>3rd submission:</strong> Run B (Replogle augmentation — potential upside)</li>
</ul>
</div>

</body></html>"""

report_path = os.path.join(PACKAGE_DIR, 'vfinal_report.html')
with open(report_path, 'w') as f:
    f.write(html)
print(f'  ✓ HTML report saved: {report_path}')

# Also save to Drive
drive_report = os.path.join(OUTPUT_DIR, f'vfinal_report_{ts_final}.html')
with open(drive_report, 'w') as f:
    f.write(html)

# === Save config as JSON ===
config_dict = {k: v for k, v in cfg.__dict__.items() if not k.startswith('_')}
config_path = os.path.join(PACKAGE_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config_dict, f, indent=2, default=str)

# Save perturbation mapping
pert_map = {all_pert_ids[i]: all_pert_symbols[i] for i in range(120)}
pert_map_path = os.path.join(PACKAGE_DIR, 'perturbation_mapping.json')
with open(pert_map_path, 'w') as f:
    json.dump(pert_map, f, indent=2)

# Save CV results
cv_path = os.path.join(PACKAGE_DIR, 'cv_results.json')
cv_export = {}
for rid in all_cv_scores:
    cv_export[f'run_{rid}'] = all_cv_scores[rid]
with open(cv_path, 'w') as f:
    json.dump(cv_export, f, indent=2)

# Save correlation results
corr_path = os.path.join(PACKAGE_DIR, 'correlation_results.json')
corr_export = {}
for rid in corr_results:
    corr_export[f'run_{rid}'] = {k: float(v) if isinstance(v, (np.floating, float)) else v
                                  for k, v in corr_results[rid].items()
                                  if not isinstance(v, dict)}
with open(corr_path, 'w') as f:
    json.dump(corr_export, f, indent=2, default=str)

# === Save final summary chart ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'V_FINAL: Submission Package Summary — {ts_final}', fontsize=15, fontweight='bold')

# Chart 1: All runs overview
ax = axes[0, 0]
rids = sorted(all_submissions.keys())
for i, rid in enumerate(rids):
    sub = all_submissions[rid]
    pub_abs = np.abs(sub.iloc[:60, 1:].values.astype(np.float32)).mean(axis=1)
    prv_abs = np.abs(sub.iloc[60:, 1:].values.astype(np.float32)).mean(axis=1)
    ax.boxplot([pub_abs, prv_abs], positions=[i*3, i*3+1], widths=0.8,
               patch_artist=True,
               boxprops=dict(facecolor=['#2196F3', '#FF9800'][0] if i == 0 else ['#4CAF50', '#FF9800'][0]))
ax.set_title('Prediction Magnitude Distribution')
labels = []
for rid in rids:
    labels.extend([f'R{rid}\npub', f'R{rid}\nprv'])
ax.set_xticks([i*3+0.5 for i in range(len(rids))])
ax.set_xticklabels([f'Run {r}' for r in rids])
ax.set_ylabel('Mean |Delta| per perturbation')

# Chart 2: Sanity check — public vs private stats should be similar
ax = axes[0, 1]
for rid in rids:
    rd = report_data[rid]
    ax.scatter(rd['pub_mean_abs'], rd['prv_mean_abs'], s=200, label=f'Run {rid}', zorder=5, edgecolors='black')
lim = max(max(rd['pub_mean_abs'], rd['prv_mean_abs']) for rd in report_data.values()) * 1.2
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='y=x (expected)')
ax.set_xlabel('Public Mean |Δ|')
ax.set_ylabel('Private Mean |Δ|')
ax.set_title('Public vs Private: Sanity Check')
ax.legend()

# Chart 3: File listing
ax = axes[1, 0]
ax.axis('off')
file_text = f'Submission Files\n{"─"*55}\n'
for rid, info in sorted(saved_files.items()):
    rd = report_data[rid]
    file_text += f'\nRun {rid}: {rd["name"]}\n'
    file_text += f'  File: {info["fname"]}\n'
    file_text += f'  CV={rd["cv_mean"]:.4f}\n'
file_text += f'\n{"─"*55}\n'
file_text += f'Package: vfinal_package_{ts_final}.zip\n'
file_text += f'Drive:   {OUTPUT_DIR}\n'
ax.text(0.02, 0.95, file_text, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.5))

# Chart 4: Historical context
ax = axes[1, 1]
hist_labels = ['V6', 'V14', 'V15.1', 'V16', 'V17', 'V18',
               'V20\nR4', 'V20\nR3★', 'V22', 'V21',
               'V23\nR5', 'V23\nR3']
hist_scores = [2.124, 3.021, 3.419, 3.584, 3.634, 3.623,
               4.033, 4.090, 4.051, 3.792, 4.064, 3.985]
colors_hist = ['#9E9E9E']*6 + ['#2196F3']*2 + ['#4CAF50']*2 + ['#E91E63']*2
ax.barh(hist_labels, hist_scores, color=colors_hist, alpha=0.8, edgecolor='white')
ax.axvline(x=4.45, color='gold', linestyle='--', linewidth=2, label='#1: 4.45')
ax.axvline(x=4.090, color='green', linestyle='--', linewidth=1.5, label='V20 best: 4.090')
ax.set_title('Historical LB Scores (public, pert_1-60)')
ax.set_xlabel('LB Score')
ax.legend(fontsize=8)
ax.set_xlim(1.5)

plt.tight_layout()
chart_path = os.path.join(PACKAGE_DIR, 'charts', 'summary.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

# === Create ZIP ===
print(f'\n--- Creating ZIP package ---')
zip_path = f'/content/vfinal_package_{ts_final}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk(PACKAGE_DIR):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, PACKAGE_DIR)
            zf.write(file_path, arcname)

zip_size = os.path.getsize(zip_path) / 1e6
print(f'  ✓ ZIP created: {zip_path} ({zip_size:.1f} MB)')

# Save ZIP to Drive too
drive_zip = os.path.join(OUTPUT_DIR, f'vfinal_package_{ts_final}.zip')
shutil.copy2(zip_path, drive_zip)
print(f'  ✓ ZIP saved to Drive: {drive_zip}')

# === Download ===
print(f'\n{"="*60}')
print('DOWNLOAD OPTIONS')
print(f'{"="*60}')
print(f'\n1. Individual submission files (for Kaggle upload):')
for rid, info in sorted(saved_files.items()):
    print(f'   Run {rid}: /content/{info["fname"]}')

print(f'\n2. Full package ZIP: {zip_path}')
print(f'\n3. All files also saved to Drive: {OUTPUT_DIR}')

print(f'\n--- Downloading ZIP ---')
try:
    colab_files.download(zip_path)
    print('  ✓ Download started')
except:
    print('  ⚠ Auto-download failed — use Files panel to download manually')

print(f'\n{"="*60}')
print('V_FINAL COMPLETE')
print(f'{"="*60}')
print(f'  Submissions: {len(saved_files)} files (Run A, B, C)')
print(f'  ALL 120 perturbations predicted (NO zeros)')
print(f'  Report + configs + charts packaged')
print(f'  Upload Run A first (safest), then Run C (ensemble)')
print(f'{"="*60}')